# To Deep Learn or Not: Accuracy, Data and Cost of Radio Map Prediction in the Upper Mid-Band

Code for the IEEE AWPL letter of the same title. The notebook generates the dataset, trains every model and produces all tables and figures of the paper. It runs on Google Colab (GPU runtime) and stores all results on Google Drive, so each phase can be interrupted and resumed.

| Phase | What it does | Output |
|---|---|---|
| **1. Scenes** | Builds 256 m × 256 m urban tiles from OpenStreetMap building footprints (city-level train/validation/test split), resolves building heights, writes Sionna meshes and a pixel-aligned height raster, and places rooftop transmitters | `scenes/` |
| **2. Ray tracing** | Computes Sionna RT path-loss maps at 7 GHz (1 m pixels, receiver height 1.5 m) and times every map. A LoS-only run and a second Monte-Carlo seed on ten scenes validate the LoS feature and measure label noise | `rt/`, `rt_checks/` |
| **3. Features** | Computes the shared 17-channel per-pixel feature stack used by every model | `feats/` |
| **4. Experiments** | Tunes every model with the same random-search budget, then trains it on every nested training set and seed, logging accuracy and cost | `exp/<fingerprint>/` |
| **5. Analysis** | Accuracy versus training data, data needed for a target RMSE, cost, covered-pixel RMSE, coverage-hole detection and an automated validity checklist | `exp/<fingerprint>/paper/` |

**Models.** All models receive the same 17 features per pixel.

| Group | Models |
|---|---|
| Classical learners (per pixel) | Ridge regression (polynomial features), k-NN, SVR, random forest (RF), LightGBM |
| Per-pixel neural network | MLP, implemented with 1×1 convolutions |
| Spatial networks | UNet-S (1.1 M parameters), UNet-M (9.8 M), Swin-UNet (12.1 M) |

**Training sizes.** Twelve nested training sets with 2, 5, 10, 20, 30, …, 100 of the 100 training tiles, i.e. 32, 80, 160, 320, …, 1600 maps (16 transmitters per tile). In the configuration they are given as fractions; the first fraction, 0.025, rounds to 2 tiles. The analysis always uses the actual number of maps logged for each run (`n_maps`, `frac_actual`).

**Hardware used for the paper.** Google Colab High-RAM runtime, 8-vCPU Intel Xeon (2.0–2.2 GHz), 50 GB RAM, NVIDIA T4 GPU (16 GB).

**Quick test.** Set `DRY_RUN = True` in the configuration cell to run the whole pipeline on procedural cities with tiny budgets on a CPU in a few minutes.


## Experimental protocol

| # | Design decision |
|---|---|
| P1 | **Identical inputs.** One 17-channel per-pixel feature stack for every model. Per-pixel models read it as a table, spatial networks as an image. All models use the same normalisation, computed on the training subset only. |
| P2 | **Only learned models.** No untrained empirical baselines. The simplest model is ridge regression on polynomial features. |
| P3 | **Five classical learners with different inductive biases** (linear, instance-based, kernel, bagged trees, boosted trees), plus a per-pixel MLP. |
| P4 | **City-level split.** Training, validation and test cities are disjoint. |
| P5 | **Data counted in ray-traced maps.** Per-pixel models sample their training pixels from the same maps as the spatial networks. |
| P6 | **Nested training sets** of 2 to 100 training tiles, with three random draws up to 30 % of the tiles and one draw above. |
| P7 | **Equal tuning budget.** Every model gets `TUNE_TRIALS` random-search trials (trial 0 is a default setting) on 30 % of the training data, selected on the validation cities. The selected settings are then kept fixed. Tuning time is counted as cost. |
| P8 | **Identical test pixels.** A fixed set of 2048 outdoor pixels per test map, used for every model. |
| P9 | **No data augmentation** in the main results (D4 augmentation is available as an optional, separately labelled run). |
| P10 | **Accuracy versus data:** RMSE for every training size with bootstrap confidence intervals over test maps, LoS/NLoS and covered-pixel breakdowns, and paired-bootstrap comparison of the best spatial and best classical model. |
| P11 | **Data versus accuracy:** minimum number of maps to reach fixed target RMSEs and the data multiplier relative to the most data-efficient classical learner. |
| P12 | **Cost:** ray-tracing and feature time, tuning and training time, inference time per map, model size, parameters, FLOPs and peak memory. |
| P13 | **Label and feature checks:** label noise from two ray-tracing seeds, and agreement of the geometric LoS feature with a LoS-only Sionna run. |
| P14 | **Pixel-sampling check:** LightGBM trained with K and 4K pixels per map must differ by less than 0.1 dB. |
| P15 | **Reproducibility:** content fingerprints in all file names, logged package versions and an automated validity checklist. |


## 0. Environment

In [ ]:
# ============================================================
# Cell 1 — Environment: installs (only if missing), imports, Drive
# ============================================================
import importlib, subprocess, sys, os

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for _pkg, _pip in [("sionna.rt", "sionna-rt"), ("osmnx", "osmnx"), ("lightgbm", "lightgbm"),
                   ("trimesh", "trimesh"), ("mapbox_earcut", "mapbox_earcut"), ("shapely", "shapely"),
                   ("sklearn", "scikit-learn"), ("scipy", "scipy"), ("pandas", "pandas"),
                   ("matplotlib", "matplotlib"), ("psutil", "psutil")]:
    _ensure(_pkg, _pip)

import json, time, math, hashlib, pickle, gc, glob, shutil, threading, warnings, platform
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import ndimage, signal
from scipy.optimize import curve_fit
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*eval_set.*")          # LightGBM API deprecation notice
warnings.filterwarnings("ignore", message=".*ill-conditioned.*")   # high-degree ridge trials (reported via RMSE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def versions():
    import sklearn, lightgbm, shapely, trimesh
    v = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
             scipy=scipy.__version__, torch=torch.__version__, sklearn=sklearn.__version__,
             lightgbm=lightgbm.__version__, shapely=shapely.__version__, trimesh=trimesh.__version__,
             device=str(DEVICE), gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"),
             cpus=os.cpu_count())
    try:
        import sionna.rt, mitsuba, drjit
        v.update(sionna_rt=getattr(sionna.rt, "__version__", "?"), mitsuba=mitsuba.__version__, drjit=drjit.__version__)
    except Exception as e:
        v["sionna_rt"] = f"unavailable ({e})"
    try:
        import osmnx
        v["osmnx"] = osmnx.__version__
    except Exception:
        pass
    return v

print(json.dumps(versions(), indent=1))


## 1. Configuration
Edit this cell only. Content-affecting settings are fingerprinted into every output filename, so changing them never mixes incompatible data.

In [ ]:
# ============================================================
# Cell 2 — Configuration (the ONLY cell you normally edit)
# ============================================================
DRY_RUN = bool(int(os.environ.get("RM_DRY_RUN", "0")))   # set True for a quick CPU test run
PRESET  = os.environ.get("RM_PRESET", "full")             # "full" (paper) | "fast" (fewer fractions/seeds/trials)

CFG = dict(
    # ---------------- phases (each is resumable) ----------------
    RUN_SCENES=True, RUN_RT=True, RUN_FEATURES=True, RUN_EXPERIMENTS=True, RUN_ANALYSIS=True,

    # ---------------- bands: each band is trained/evaluated separately (in-distribution) -----
    BANDS_GHZ=[7.0],              # carrier frequency (GHz) used in the paper

    # ---------------- scenes ----------------
    SCENE_SOURCE="osm",           # "osm" (paper) | "procedural" (dry run / offline)
    # city -> (lat, lon, split). City-level split: no city appears in two splits.
    CITIES={
        "london":    (51.5136,  -0.0890, "train"), "paris":     (48.8606,   2.3376, "train"),
        "berlin":    (52.5200,  13.4050, "train"), "madrid":    (40.4168,  -3.7038, "train"),
        "chicago":   (41.8800, -87.6300, "train"), "toronto":   (43.6510, -79.3810, "train"),
        "tokyo":     (35.6812, 139.7671, "train"), "singapore": ( 1.2840, 103.8510, "train"),
        "sydney":    (-33.8688, 151.2093, "train"), "vienna":   (48.2082,  16.3738, "train"),
        "barcelona": (41.3874,   2.1686, "val"),   "sanfrancisco": (37.7900, -122.4000, "val"),
        "newyork":   (40.7549, -73.9840, "test"),  "hongkong":  (22.2800, 114.1600, "test"),
        "munich":    (48.1374,  11.5755, "test"),
    },
    TILES_PER_CITY=10,
    TILE_M=256.0, CELL_M=1.0,     # 256 x 256 pixels at 1 m
    TILE_MARGIN_M=40.0,           # buildings within this margin are meshed (scattering) but not mapped
    TILE_SEARCH_RADIUS_M=1500.0, TILE_MIN_SEP_M=300.0, MAX_TILE_TRIES=400,
    MIN_BLDG_COVER=0.10, MAX_BLDG_COVER=0.70,
    MIN_BLDG_AREA_M2=6.0, POLY_SIMPLIFY_M=0.25,
    LEVEL_HEIGHT_M=3.0, DEFAULT_HEIGHT_M=9.0, MIN_VALID_HEIGHT_M=2.0, MAX_VALID_HEIGHT_M=450.0,
    BUILDING_MATERIAL="concrete", BUILDING_THICKNESS_M=0.3, GROUND_MATERIAL="medium_dry_ground",
    RX_H=1.5,
    N_TX_PER_SCENE=16, TX_ABOVE_ROOF_M=3.0, TX_MIN_ROOF_M=6.0, TX_CENTRAL_FRAC=0.75,
    TX_MIN_SEP_M=15.0, TX_ROOF_EDGE_BUFFER_M=1.0, TX_ROOF_EDGE_MAX_M=4.0,

    # ---------------- ray tracing (Sionna RT RadioMapSolver) ----------------
    RT=dict(max_depth=5, samples_per_tx=int(3e7), los=True, specular_reflection=True,
            diffuse_reflection=False, refraction=False, diffraction=True, edge_diffraction=False,
            diffraction_lit_region=True, seed=7),
    TX_BATCH=1,                   # transmitters per solver call (raise only if GPU memory allows)
    # Label policy for outdoor cells that no ray reaches (and very weak cells):
    #  "floor": PL clipped at PL_FLOOR_DB, unhit cells = floor ("out of coverage"); every outdoor cell is
    #           labelled and no map is dropped (no selection bias). Standard truncation (RadioMapSeer-style).
    #  "mask":  unhit cells excluded from loss and metrics for all models; maps with > MAX_INVALID_FRAC
    #           unhit outdoor cells dropped (logged).
    LABEL_MODE="floor", PL_FLOOR_DB=160.0, MAX_INVALID_FRAC=0.20,
    LOS_CHECK_SCENES=10, LOS_CHECK_SAMPLES=int(2e7),
    NOISE_CHECK_SCENES=10,

    # ---------------- features ----------------
    FEATURE_SET=os.environ.get("RM_FEATURE_SET", "B"),  # "B" = all 17 channels (paper); "A" = reduced 7-channel set (optional)
    RAY_STEP_PX=0.5, RAY_RASTER_UPSAMPLE=2, CONTEXT_RADII_M=(10, 25, 50), CANYON_HALF_WIDTH_M=20.0,

    # ---------------- experiment protocol ----------------
    # Fractions of the 100 training tiles. 0.025 x 100 = 2.5 rounds to 2 tiles (32 maps); the analysis
    # always uses the actual number of maps logged for each run (n_maps, frac_actual).
    FRACTIONS=[0.025, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    SMALL_FRAC_MAX=0.3, SEEDS_SMALL=[0, 1, 2], SEEDS_LARGE=[0],
    K_PIX=1500, K_CHECK_MULT=4,   # classical: pixels sampled per training map; sufficiency check at 4K
    EVAL_PIX_PER_MAP=2048, EVAL_PIX_SEED=2027, NORM_PIX_PER_MAP=4096,
    KNN_CAP=50_000, SVR_CAP=10_000, RF_N_EST=100, RF_MAX_SAMPLES=200_000,
    FULL_EVAL_SKIP=("knn", "svr"),  # supplementary full-map metrics skipped for these (cost)
    MODELS=["ridge", "knn", "svr", "rf", "lgbm", "mlp1x1", "unet_s", "unet_m", "swin_unet"],
    OPTIONAL_MODELS=[],            # e.g. ["unet_shallow"] for the receptive-field ablation
    TUNE_FRACTION=0.3, TUNE_SEED=0, TUNE_TRIALS=6,   # SAME number of trials for every model
    BATCH=8, MAX_STEPS=12_000, MIN_STEPS=1_500, EPOCHS_EQ=60, EVAL_EVERY=None, PATIENCE=8,
    VAL_MAPS_MAX=200, AMP=True, NUM_WORKERS=2,
    AUGMENT_D4=False,             # keep False for the main results (protocol P9)
    RUN_AUG_ABLATION=False,       # adds DL rows with D4 augmentation, labelled separately
    BOOTSTRAP=1000, CI=0.95, EPS3_MARGIN_DB=0.5, K_SUFFICIENCY_DB=0.1, LOS_AGREEMENT_MIN=0.95,
)

if PRESET == "fast" and not DRY_RUN:
    CFG.update(FRACTIONS=[0.025, 0.05, 0.1, 0.3, 0.5, 0.7, 1.0], SEEDS_SMALL=[0, 1], TUNE_TRIALS=4)

if DRY_RUN:   # tiny, CPU-friendly smoke test of every code path
    CFG.update(
        SCENE_SOURCE="procedural",
        CITIES={"synA": (0, 0, "train"), "synB": (0, 0, "train"), "synC": (0, 0, "train"),
                "synD": (0, 0, "train"), "synE": (0, 0, "val"), "synF": (0, 0, "test")},
        TILES_PER_CITY=2, TILE_M=64.0, TILE_MARGIN_M=8.0, TILE_MIN_SEP_M=70.0, N_TX_PER_SCENE=3, TX_MIN_SEP_M=8.0,
        RT=dict(CFG["RT"], max_depth=3, samples_per_tx=int(3e5)), TX_BATCH=3,
        LOS_CHECK_SCENES=2, LOS_CHECK_SAMPLES=int(2e6), NOISE_CHECK_SCENES=2, MAX_INVALID_FRAC=0.6,
        FRACTIONS=[0.25, 0.5, 1.0], SMALL_FRAC_MAX=0.5, SEEDS_SMALL=[0, 1], SEEDS_LARGE=[0],
        K_PIX=300, EVAL_PIX_PER_MAP=256, NORM_PIX_PER_MAP=512, KNN_CAP=3000, SVR_CAP=1500,
        RF_N_EST=20, RF_MAX_SAMPLES=5000, TUNE_FRACTION=0.5, TUNE_TRIALS=2,
        BATCH=4, MAX_STEPS=30, MIN_STEPS=20, EPOCHS_EQ=2, EVAL_EVERY=10, PATIENCE=3,
        NUM_WORKERS=0, BOOTSTRAP=100, OPTIONAL_MODELS=["unet_shallow"], RUN_AUG_ABLATION=True,
    )

# ---------------- paths ----------------
if IN_COLAB:
    ROOT = Path(os.environ.get("RM_DRIVE_ROOT", "/content/drive/MyDrive/rm_ml_vs_dl") + ("_dryrun" if DRY_RUN else ""))
    LOCAL_CACHE = Path("/content/rm_cache")
else:
    ROOT = Path(os.environ.get("RM_ROOT", "./rm_ml_vs_dl" + ("_dryrun" if DRY_RUN else ""))).resolve()
    LOCAL_CACHE = ROOT / "_local_cache"
for _d in ["scenes", "rt", "rt_checks", "feats", "exp", "osm_cache", "logs"]:
    (ROOT / _d).mkdir(parents=True, exist_ok=True)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# ---------------- fingerprints (content-affecting knobs only) ----------------
def fingerprint(d, n=6):
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:n]

# Scope knobs (number of cities/tiles/transmitters, list of bands) are deliberately NOT in the
# fingerprints: widening the scope extends the dataset instead of orphaning it.
SCENE_FP = fingerprint({k: CFG[k] for k in [
    "SCENE_SOURCE", "TILE_M", "CELL_M", "TILE_MARGIN_M", "TILE_SEARCH_RADIUS_M", "TILE_MIN_SEP_M",
    "MIN_BLDG_COVER", "MAX_BLDG_COVER", "MIN_BLDG_AREA_M2", "POLY_SIMPLIFY_M", "LEVEL_HEIGHT_M",
    "DEFAULT_HEIGHT_M", "MIN_VALID_HEIGHT_M", "MAX_VALID_HEIGHT_M", "TX_ABOVE_ROOF_M", "TX_MIN_ROOF_M",
    "TX_CENTRAL_FRAC", "TX_MIN_SEP_M", "TX_ROOF_EDGE_BUFFER_M", "TX_ROOF_EDGE_MAX_M", "BUILDING_MATERIAL",
    "BUILDING_THICKNESS_M", "GROUND_MATERIAL", "RAY_RASTER_UPSAMPLE"]})
RT_FP = fingerprint(dict(scene=SCENE_FP, rt=CFG["RT"], rx_h=CFG["RX_H"]))
FEAT_VERSION = "v1"
FEAT_FP = fingerprint(dict(scene=SCENE_FP, v=FEAT_VERSION, step=CFG["RAY_STEP_PX"],
                           radii=CFG["CONTEXT_RADII_M"], canyon=CFG["CANYON_HALF_WIDTH_M"], rx_h=CFG["RX_H"]))
LABEL_TAG = f"{CFG['LABEL_MODE']}{CFG['PL_FLOOR_DB']:g}" if CFG["LABEL_MODE"] == "floor" else f"mask{CFG['MAX_INVALID_FRAC']:g}"
EXP_FP = fingerprint(dict(rt=RT_FP, feat=FEAT_FP, fs=CFG["FEATURE_SET"], label=LABEL_TAG, **{k: CFG[k] for k in [
    "K_PIX", "EVAL_PIX_PER_MAP", "EVAL_PIX_SEED", "NORM_PIX_PER_MAP", "KNN_CAP", "SVR_CAP", "RF_N_EST",
    "RF_MAX_SAMPLES", "TUNE_FRACTION", "TUNE_SEED", "TUNE_TRIALS", "BATCH", "MAX_STEPS", "MIN_STEPS",
    "EPOCHS_EQ", "EVAL_EVERY", "PATIENCE", "VAL_MAPS_MAX", "AMP"]}))
EXP_DIR = ROOT / "exp" / f"exp_{EXP_FP}"
PAPER_DIR = EXP_DIR / "paper"
for _d in [EXP_DIR, PAPER_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

N_PIX = int(round(CFG["TILE_M"] / CFG["CELL_M"]))
SPLIT_OF_CITY = {c: v[2] for c, v in CFG["CITIES"].items()}
json.dump(dict(CFG=CFG, SCENE_FP=SCENE_FP, RT_FP=RT_FP, FEAT_FP=FEAT_FP, EXP_FP=EXP_FP,
               DRY_RUN=DRY_RUN, PRESET=PRESET, versions=versions()),
          open(EXP_DIR / "config_snapshot.json", "w"), indent=1, default=str)
print(f"ROOT={ROOT}\nDRY_RUN={DRY_RUN} PRESET={PRESET} N_PIX={N_PIX}\n"
      f"SCENE_FP={SCENE_FP} RT_FP={RT_FP} FEAT_FP={FEAT_FP} EXP_FP={EXP_FP}")


## 2. Phase 1 — Scenes
OSM footprints → disjoint polygons (the taller building wins overlaps) → Sionna meshes **and** a height raster sampled at exactly the radio-map cell centres. Building heights come from `height`, then `building:levels` × 3 m, then city-empirical imputation; the source of every height is logged. Transmitters sit on rooftops (roof + 3 m) inside the central 75 % of the tile.

In [ ]:
# ============================================================
# Cell 3 — Phase 1: scenes (OSM or procedural) -> meshes + aligned height raster + Tx sites
# ============================================================
import shapely
from shapely.geometry import Polygon, MultiPolygon, box, Point
from shapely.ops import unary_union
from shapely.strtree import STRtree
import trimesh

T, C, MARGIN = CFG["TILE_M"], CFG["CELL_M"], CFG["TILE_MARGIN_M"]

def grid_centers(n=N_PIX, tile=T, cell=C):
    """Cell centres in the Sionna PlanarRadioMap convention: row i -> y, col j -> x, both increasing."""
    c = -tile / 2 + (np.arange(n) + 0.5) * cell
    X, Y = np.meshgrid(c, c)          # X[i, j] = x_j, Y[i, j] = y_i
    return X, Y

# ---------- building heights: parse OSM tags, then impute with logging ----------
def _num(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = str(s).strip().lower().replace(",", ".")
    feet = ("ft" in s) or ("'" in s)
    for tok in ["m", "ft", "'", '"', "meters", "metres"]:
        s = s.replace(tok, " ")
    try:
        v = float(s.split()[0].split(";")[0])
    except Exception:
        return None
    return v * 0.3048 if feet else v

def height_from_tags(tags):
    lo, hi = CFG["MIN_VALID_HEIGHT_M"], CFG["MAX_VALID_HEIGHT_M"]
    for key in ("height", "building:height"):
        v = _num(tags.get(key))
        if v is not None and lo <= v <= hi:
            return v, "height"
    lv = _num(tags.get("building:levels"))
    if lv is not None and 0 < lv < 200:
        v = lv * CFG["LEVEL_HEIGHT_M"]
        if lo <= v <= hi:
            return v, "levels"
    return None, None

def resolve_heights(tag_list, rng):
    """Tagged heights first; missing ones drawn from the city's empirical height distribution
    (or a default when fewer than 30 buildings carry a usable tag). Every source is logged."""
    hs, src = [], []
    for t in tag_list:
        h, s = height_from_tags(t)
        hs.append(h); src.append(s)
    known = np.array([h for h in hs if h is not None], float)
    for i, h in enumerate(hs):
        if h is None:
            if len(known) >= 30:
                hs[i], src[i] = float(rng.choice(known)), "imputed_empirical"
            else:
                hs[i], src[i] = CFG["DEFAULT_HEIGHT_M"], "imputed_default"
    return np.array(hs, float), src

# ---------- geometry helpers ----------
def _polys(g):
    if g is None or g.is_empty:
        return []
    if isinstance(g, Polygon):
        return [g]
    if isinstance(g, MultiPolygon):
        return list(g.geoms)
    if hasattr(g, "geoms"):
        return [p for gg in g.geoms for p in _polys(gg)]
    return []

def clean_polygon(p):
    if not p.is_valid:
        p = shapely.make_valid(p)
    out = []
    for q in _polys(p):
        q = q.simplify(CFG["POLY_SIMPLIFY_M"], preserve_topology=True)
        if q.is_valid and q.area >= CFG["MIN_BLDG_AREA_M2"]:
            out.append(shapely.geometry.polygon.orient(q, 1.0))
    return out

def make_disjoint(polys, heights, srcs):
    """Overlapping footprints -> disjoint; the taller building keeps the shared area.
    Guarantees the mesh and the raster describe exactly the same geometry."""
    order = np.argsort(-np.asarray(heights))
    taken, P, H_, S_ = None, [], [], []
    for i in order:
        g = polys[i] if taken is None else polys[i].difference(taken)
        for q in clean_polygon(g) if not g.is_empty else []:
            P.append(q); H_.append(float(heights[i])); S_.append(srcs[i])
        taken = polys[i] if taken is None else unary_union([taken, polys[i]])
    return P, np.array(H_), S_

def rasterize(polys, heights, up=1):
    """Height raster sampled at cell centres. up=1: the radio-map grid; up>1: a finer grid used only by
    the ray walk of the obstruction features (reduces roof-edge discretisation error)."""
    n, cell = N_PIX * up, C / up
    X, Y = grid_centers(n, T, cell)
    H = np.zeros((n, n), np.float32)
    c0 = -T / 2 + 0.5 * cell
    for p, h in zip(polys, heights):
        x0, y0, x1, y1 = p.bounds
        j0, j1 = max(0, int((x0 - c0) // cell)), min(n - 1, int((x1 - c0) // cell) + 1)
        i0, i1 = max(0, int((y0 - c0) // cell)), min(n - 1, int((y1 - c0) // cell) + 1)
        if j1 < j0 or i1 < i0:
            continue
        sub = shapely.contains_xy(p, X[i0:i1 + 1, j0:j1 + 1], Y[i0:i1 + 1, j0:j1 + 1])
        H[i0:i1 + 1, j0:j1 + 1] = np.where(sub, np.maximum(H[i0:i1 + 1, j0:j1 + 1], h), H[i0:i1 + 1, j0:j1 + 1])
    return H

def write_sionna_scene(sdir, polys, heights):
    (sdir / "meshes").mkdir(parents=True, exist_ok=True)
    meshes = [trimesh.creation.extrude_polygon(p, float(h)) for p, h in zip(polys, heights)]
    trimesh.util.concatenate(meshes).export(sdir / "meshes" / "bldgs.ply")
    g = T / 2 + MARGIN + 10.0
    trimesh.Trimesh(vertices=[[-g, -g, 0], [g, -g, 0], [g, g, 0], [-g, g, 0]],
                    faces=[[0, 1, 2], [0, 2, 3]]).export(sdir / "meshes" / "terrain.ply")
    xml = f"""<scene version="2.1.0">
  <bsdf type="itu-radio-material" id="rm_bldg"><string name="type" value="{CFG['BUILDING_MATERIAL']}"/><float name="thickness" value="{CFG['BUILDING_THICKNESS_M']}"/></bsdf>
  <bsdf type="itu-radio-material" id="rm_ground"><string name="type" value="{CFG['GROUND_MATERIAL']}"/><float name="thickness" value="1.0"/></bsdf>
  <shape type="ply" id="mesh-bldgs"><string name="filename" value="meshes/bldgs.ply"/><boolean name="face_normals" value="true"/><ref id="rm_bldg" name="bsdf"/></shape>
  <shape type="ply" id="mesh-terrain"><string name="filename" value="meshes/terrain.ply"/><boolean name="face_normals" value="true"/><ref id="rm_ground" name="bsdf"/></shape>
</scene>
"""
    (sdir / "scene.xml").write_text(xml)

def place_transmitters(polys, heights, rng, n_tx):
    """Rooftop sites: Tx = roof + TX_ABOVE_ROOF_M, between TX_ROOF_EDGE_BUFFER_M and TX_ROOF_EDGE_MAX_M from the
    roof edge (antennas near the parapet; a site deep inside a large roof is shadowed by its own roof),
    inside the central TX_CENTRAL_FRAC of the tile, mutually >= TX_MIN_SEP_M apart.
    Sequential draws from a seeded RNG: raising N_TX later keeps the first sites unchanged."""
    half = CFG["TX_CENTRAL_FRAC"] * T / 2
    cand = []
    for p, h in zip(polys, heights):
        if h < CFG["TX_MIN_ROOF_M"]:
            continue
        inner = p.buffer(-CFG["TX_ROOF_EDGE_BUFFER_M"])
        ring = inner.difference(p.buffer(-CFG["TX_ROOF_EDGE_MAX_M"]))   # near the parapet, like real rooftop sites
        q = (ring if not ring.is_empty else inner).intersection(box(-half, -half, half, half))
        if not q.is_empty and q.area > 1.0:
            cand.append((q, h))
    if not cand:
        return None
    w = np.array([q.area for q, _ in cand]); w = w / w.sum()
    sites, tries = [], 0
    while len(sites) < n_tx and tries < 200 * n_tx:
        tries += 1
        q, h = cand[rng.choice(len(cand), p=w)]
        x0, y0, x1, y1 = q.bounds
        x, y = rng.uniform(x0, x1), rng.uniform(y0, y1)
        if not q.contains(Point(x, y)):
            continue
        if any((x - s[0]) ** 2 + (y - s[1]) ** 2 < CFG["TX_MIN_SEP_M"] ** 2 for s in sites):
            continue
        sites.append([float(x), float(y), float(h + CFG["TX_ABOVE_ROOF_M"])])
    return sites if len(sites) == n_tx else None

# ---------- building sources ----------
OVERPASS_ENDPOINTS = [            # tried in order; the main server often refuses/limits shared Colab IPs
    "https://overpass-api.de/api",
    "https://lz4.overpass-api.de/api",
    "https://z.overpass-api.de/api",
    "https://overpass.kumi.systems/api",
    "https://overpass.private.coffee/api",
    "https://maps.mail.ru/osm/tools/overpass/api",
]

def _fetch_buildings(lat, lon, dist, rounds=3):
    """Download building footprints with endpoint rotation and exponential back-off."""
    import osmnx as ox
    ox.settings.requests_timeout = 600                          # (osmnx 2.x name; 'timeout' is ignored)
    ox.settings.overpass_memory = 2 * 1024 ** 3                 # dense downtowns need a larger server-side budget
    ox.settings.use_cache = True
    ox.settings.cache_folder = str(ROOT / "osm_cache" / "http")  # on Drive: survives runtime restarts
    last = None
    for rnd in range(rounds):
        for ep in OVERPASS_ENDPOINTS:
            ox.settings.overpass_url = ep
            ox.settings.overpass_rate_limit = ep.startswith("https://overpass-api.de")  # only main has /status
            try:
                t0 = time.time()
                gdf = ox.features_from_point((lat, lon), tags={"building": True}, dist=dist)
                print(f"   OSM: {len(gdf)} features from {ep} in {time.time() - t0:.0f}s")
                return gdf
            except Exception as e:                                # refused / timeout / 429 / 504
                last = e
                print(f"   OSM: {ep} failed ({type(e).__name__}: {str(e)[:90]})")
        wait = 60 * 2 ** rnd
        print(f"   OSM: all endpoints failed (round {rnd + 1}/{rounds}); waiting {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"Overpass unreachable from this runtime after {rounds} rounds: {last}. "
                       "Re-run later (finished cities are cached), or restart the runtime to get a new IP.")

def osm_city(city, lat, lon):
    """All building footprints within the search radius, in a local metric frame centred on the city."""
    cache = ROOT / "osm_cache" / f"{city}.pkl"
    if cache.exists():
        return pickle.load(open(cache, "rb"))
    dist = CFG["TILE_SEARCH_RADIUS_M"] + T / 2 + MARGIN + 50
    gdf = _fetch_buildings(lat, lon, dist)
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    crs = gdf.estimate_utm_crs()
    gdf = gdf.to_crs(crs)
    import geopandas as gpd
    cpt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(crs).iloc[0]
    keys = [k for k in ("height", "building:height", "building:levels") if k in gdf.columns]
    polys, tags = [], []
    for _, row in gdf.iterrows():
        t = {k: row[k] for k in keys if isinstance(row[k], str) or (row[k] is not None and not pd.isna(row[k]))}
        for p in _polys(row.geometry):
            polys.append(shapely.affinity.translate(p, -cpt.x, -cpt.y)); tags.append(t)
    out = dict(polys=polys, tags=tags, crs=str(crs), center_xy=(cpt.x, cpt.y))
    pickle.dump(out, open(cache, "wb"))
    return out

def procedural_city(city):
    """Synthetic city with its own morphology (block size, street width, height distribution, tag
    completeness). Used for DRY_RUN and offline tests; exercises the same parsing/imputation path."""
    seed = int(hashlib.md5(city.encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    blk, street = rng.uniform(18, 40), rng.uniform(6, 14)
    mu, sig, p_tag = rng.uniform(2.2, 3.2), rng.uniform(0.3, 0.6), rng.uniform(0.4, 0.9)
    R = CFG["TILE_SEARCH_RADIUS_M"] if not DRY_RUN else 3 * T
    polys, tags = [], []
    xs = np.arange(-R, R, blk + street)
    for x0 in xs:
        for y0 in xs:
            if rng.random() < 0.15:          # open space / park
                continue
            nx, ny = rng.integers(1, 3, 2)
            w, l = blk / nx, blk / ny
            for a in range(nx):
                for b in range(ny):
                    sx, sy = rng.uniform(0, 0.25 * w), rng.uniform(0, 0.25 * l)
                    p = box(x0 + a * w + sx, y0 + b * l + sy, x0 + (a + 1) * w - sx * 0.3, y0 + (b + 1) * l - sy * 0.3)
                    h = float(np.clip(rng.lognormal(mu, sig), 4, 120))
                    u = rng.random()
                    t = ({"height": f"{h:.1f} m"} if u < p_tag * 0.6 else
                         {"building:levels": str(max(1, int(round(h / 3))))} if u < p_tag else {})
                    polys.append(p); tags.append(t)
    return dict(polys=polys, tags=tags, crs="local", center_xy=(0.0, 0.0))

# ---------- tile selection ----------
def build_city_tiles(city, lat, lon, split, n_tiles):
    src = osm_city(city, lat, lon) if CFG["SCENE_SOURCE"] == "osm" else procedural_city(city)
    rng = np.random.default_rng(int(hashlib.md5(f"{city}|{SCENE_FP}".encode()).hexdigest()[:8], 16))
    heights_all, srcs_all = resolve_heights(src["tags"], rng)
    tree = STRtree(src["polys"])
    R = CFG["TILE_SEARCH_RADIUS_M"] if not DRY_RUN else 2 * T
    accepted, out, rejects = [], [], dict(cover=0, sep=0, tx=0)
    for attempt in range(CFG["MAX_TILE_TRIES"]):
        if len(out) >= n_tiles:
            break
        r, th = R * np.sqrt(rng.random()), rng.uniform(0, 2 * np.pi)
        cx, cy = r * np.cos(th), r * np.sin(th)
        if any((cx - a) ** 2 + (cy - b) ** 2 < CFG["TILE_MIN_SEP_M"] ** 2 for a, b in accepted):
            rejects["sep"] += 1; continue
        big = box(cx - T / 2 - MARGIN, cy - T / 2 - MARGIN, cx + T / 2 + MARGIN, cy + T / 2 + MARGIN)
        idx = tree.query(big)
        P, Hh, S = [], [], []
        for i in idx:
            g = src["polys"][i].intersection(big)
            for q in _polys(g):
                P.append(shapely.affinity.translate(q, -cx, -cy)); Hh.append(heights_all[i]); S.append(srcs_all[i])
        P, Hh, S = make_disjoint(P, Hh, S) if P else ([], np.array([]), [])
        tile_box = box(-T / 2, -T / 2, T / 2, T / 2)
        inside = [k for k, p in enumerate(P) if p.intersects(tile_box)]
        cover = sum(P[k].intersection(tile_box).area for k in inside) / T ** 2
        if not (CFG["MIN_BLDG_COVER"] <= cover <= CFG["MAX_BLDG_COVER"]):
            rejects["cover"] += 1; continue
        in_tile = [(P[k].intersection(tile_box), Hh[k]) for k in inside]
        in_tile = [(q, h) for g, h in in_tile for q in _polys(g)]
        sites = place_transmitters([q for q, _ in in_tile], [h for _, h in in_tile], rng, CFG["N_TX_PER_SCENE"])
        if sites is None:
            rejects["tx"] += 1; continue
        accepted.append((cx, cy))
        n_imp = sum(S[k].startswith("imputed") for k in inside)
        out.append(dict(city=city, split=split, idx=len(out), center_local=(cx, cy),
                        center_proj=(src["center_xy"][0] + cx, src["center_xy"][1] + cy), crs=src["crs"],
                        polys=P, heights=Hh, sources=S, tx=sites, cover=cover,
                        n_bldg_tile=len(inside), n_imputed_tile=int(n_imp)))
    return out, rejects

def build_all_scenes():
    manifest = []
    for city, (lat, lon, split) in CFG["CITIES"].items():
        done = sorted(glob.glob(str(ROOT / "scenes" / f"{city}_*_{SCENE_FP}" / "meta.json")))
        if len(done) >= CFG["TILES_PER_CITY"]:
            manifest += [json.load(open(p)) for p in done[:CFG["TILES_PER_CITY"]]]
            continue
        t0 = time.time()
        tiles, rejects = build_city_tiles(city, lat, lon, split, CFG["TILES_PER_CITY"])
        for t in tiles:
            sid = f"{city}_{t['idx']:02d}"
            sdir = ROOT / "scenes" / f"{sid}_{SCENE_FP}"
            sdir.mkdir(parents=True, exist_ok=True)
            write_sionna_scene(sdir, t["polys"], t["heights"])
            tile_box = box(-T / 2, -T / 2, T / 2, T / 2)
            pin = [(p, h) for p, h in zip(t["polys"], t["heights"]) if p.intersects(tile_box)]
            H = rasterize([p for p, _ in pin], [h for _, h in pin])
            Hf = rasterize([p for p, _ in pin], [h for _, h in pin], up=CFG["RAY_RASTER_UPSAMPLE"])
            np.savez_compressed(sdir / "scene.npz", H=H, bmask=(H > 0), H_fine=Hf)
            # Every Tx must sit on a roof in the raster too (mesh/raster consistency).
            for x, y, z in t["tx"]:
                j, i = int((x + T / 2) / C), int((y + T / 2) / C)
                assert H[i, j] > 0 and abs(H[i, j] + CFG["TX_ABOVE_ROOF_M"] - z) < 1e-3, "Tx/raster mismatch"
            meta = dict(scene_id=sid, city=city, split=split, scene_fp=SCENE_FP, dir=str(sdir),
                        center_local=t["center_local"], center_proj=t["center_proj"], crs=t["crs"],
                        tx=t["tx"], cover=float(t["cover"]), n_bldg_tile=t["n_bldg_tile"],
                        n_imputed_tile=t["n_imputed_tile"],
                        imputed_frac=t["n_imputed_tile"] / max(1, t["n_bldg_tile"]),
                        height_median=float(np.median(H[H > 0])), height_p90=float(np.percentile(H[H > 0], 90)))
            json.dump(meta, open(sdir / "meta.json", "w"), indent=1)
            manifest.append(meta)
        print(f"{city:>13s} [{split}] tiles={len(tiles)}/{CFG['TILES_PER_CITY']} rejects={rejects} "
              f"({time.time() - t0:.1f}s)")
        if len(tiles) < CFG["TILES_PER_CITY"]:
            print(f"   WARNING: only {len(tiles)} tiles for {city}; widen TILE_SEARCH_RADIUS_M or relax cover limits")
    df = pd.DataFrame(manifest)
    df.to_csv(ROOT / "scenes" / f"manifest_{SCENE_FP}.csv", index=False)
    return df

if CFG["RUN_SCENES"]:
    SCENES = build_all_scenes()
else:
    SCENES = pd.read_csv(ROOT / "scenes" / f"manifest_{SCENE_FP}.csv")
SCENES["tx"] = SCENES["tx"].apply(lambda v: v if isinstance(v, list) else json.loads(v))
print(SCENES.groupby("split").agg(scenes=("scene_id", "count"), cities=("city", "nunique"),
                                  cover=("cover", "mean"), imputed=("imputed_frac", "mean")))

## 3. Phase 2 — Ray tracing
One Sionna `RadioMapSolver` call per transmitter. The raw path gains are stored, and the label policy is applied when the data are loaded. With the policy used in the paper (`LABEL_MODE = "floor"`), outdoor pixels that no path reaches, and pixels with a path loss above 160 dB, are set to 160 dB, and no map is excluded. The alternative `LABEL_MODE = "mask"` excludes unreached pixels instead. Ten scenes also get a LoS-only run, to validate the geometric LoS feature, and a second Monte-Carlo seed, to measure label noise.


In [ ]:
# ============================================================
# Cell 4 — Phase 2: Sionna RT path-loss maps (+ LoS-only and label-noise check runs)
# ============================================================
import inspect

def band_tag(f_ghz):
    return f"{f_ghz:g}GHz"

def _solver_kwargs(solver, kw):
    ok = set(inspect.signature(solver.__call__).parameters)
    dropped = [k for k in kw if k not in ok]
    if dropped:
        print(f"   NOTE: this Sionna version ignores {dropped}")
    return {k: v for k, v in kw.items() if k in ok}

def run_rt(meta, f_ghz, rt_kw, check_alignment=True):
    """Returns path gain [n_tx, N, N] (linear) and seconds per map."""
    from sionna.rt import load_scene, PlanarArray, Transmitter, RadioMapSolver
    scene = load_scene(str(Path(meta["dir"]) / "scene.xml"))
    scene.frequency = f_ghz * 1e9
    arr = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5,
                      pattern="iso", polarization="V")
    scene.tx_array, scene.rx_array = arr, arr
    solver = RadioMapSolver()
    kw = _solver_kwargs(solver, rt_kw)
    tx = meta["tx"]
    pg, secs = [], []
    for b in range(0, len(tx), CFG["TX_BATCH"]):
        for name in list(scene.transmitters.keys()):
            scene.remove(name)
        batch = tx[b:b + CFG["TX_BATCH"]]
        for k, (x, y, z) in enumerate(batch):
            scene.add(Transmitter(name=f"tx{b + k}", position=[x, y, z]))
        t0 = time.time()
        rm = solver(scene, center=[0.0, 0.0, CFG["RX_H"]], orientation=[0.0, 0.0, 0.0],
                    size=[T, T], cell_size=[C, C], **kw)
        g = np.asarray(rm.path_gain.numpy(), np.float64)     # .numpy() forces evaluation
        secs += [(time.time() - t0) / len(batch)] * len(batch)
        if check_alignment and b == 0:
            cc = np.asarray(rm.cell_centers.numpy())
            X, Y = grid_centers()
            assert cc.shape[:2] == (N_PIX, N_PIX), cc.shape
            assert np.allclose(cc[..., 0], X, atol=1e-3) and np.allclose(cc[..., 1], Y, atol=1e-3), \
                "Sionna cell grid is not aligned with the height raster"
        pg.append(g)
        del rm
    del scene
    gc.collect()
    try:
        import drjit as dr
        dr.flush_malloc_cache()
    except Exception:
        pass
    return np.concatenate(pg, 0), np.array(secs)

def pick_check_scenes(n):
    """Deterministic, spread across splits and cities."""
    s = SCENES.sort_values(["split", "city", "scene_id"]).reset_index(drop=True)
    idx = np.unique(np.linspace(0, len(s) - 1, max(1, n)).round().astype(int))
    return s.iloc[idx]["scene_id"].tolist()

def rt_phase():
    rows = []
    los_ids = set(pick_check_scenes(CFG["LOS_CHECK_SCENES"]))
    noise_ids = set(pick_check_scenes(CFG["NOISE_CHECK_SCENES"]))
    for f in CFG["BANDS_GHZ"]:
        bdir = ROOT / "rt" / band_tag(f); bdir.mkdir(exist_ok=True)
        cdir = ROOT / "rt_checks" / band_tag(f); cdir.mkdir(exist_ok=True)
        t_band = time.time()
        for _, m in SCENES.iterrows():
            sid = m["scene_id"]
            fn = bdir / f"{sid}_{RT_FP}.npz"
            if not fn.exists():
                pg, secs = run_rt(m, f, CFG["RT"])
                bm = np.load(Path(m["dir"]) / "scene.npz")["bmask"]
                valid = (pg > 0) & (~bm[None])
                pl = np.where(pg > 0, -10 * np.log10(np.maximum(pg, 1e-300)), np.nan).astype(np.float32)
                np.savez_compressed(fn, pl_db=pl, valid=valid, outdoor=~bm, rt_s=secs)   # raw labels; policy applied at load
            d = np.load(fn)
            bm = np.load(Path(m["dir"]) / "scene.npz")["bmask"]
            n_out = (~bm).sum()
            for k in range(d["pl_db"].shape[0]):
                inv = 1.0 - d["valid"][k].sum() / max(1, n_out)
                weak = float(((d["pl_db"][k] >= CFG["PL_FLOOR_DB"]) & d["valid"][k]).sum() / max(1, n_out))
                rows.append(dict(scene_id=sid, city=m["city"], split=m["split"], tx_idx=k, band=f,
                                 rt_file=str(fn), invalid_frac=float(inv), rt_s=float(d["rt_s"][k]),
                                 floor_frac=float(inv + weak) if CFG["LABEL_MODE"] == "floor" else 0.0,
                                 include=bool(CFG["LABEL_MODE"] == "floor" or inv <= CFG["MAX_INVALID_FRAC"]),
                                 pl_median=float(np.nanmedian(d["pl_db"][k][d["valid"][k]]))))
            # LoS-only pass (validates the geometric LoS feature)
            if sid in los_ids and not (cdir / f"los_{sid}_{RT_FP}.npz").exists():
                kw = dict(CFG["RT"], max_depth=0, los=True, specular_reflection=False, diffuse_reflection=False,
                          refraction=False, diffraction=False, edge_diffraction=False,
                          samples_per_tx=CFG["LOS_CHECK_SAMPLES"])
                pg0, _ = run_rt(m, f, kw, check_alignment=False)
                np.savez_compressed(cdir / f"los_{sid}_{RT_FP}.npz", los=(pg0 > 0))
            # Second Monte-Carlo seed (label-noise floor)
            if sid in noise_ids and not (cdir / f"noise_{sid}_{RT_FP}.npz").exists():
                pg2, _ = run_rt(m, f, dict(CFG["RT"], seed=CFG["RT"]["seed"] + 1), check_alignment=False)
                pl2 = np.where(pg2 > 0, -10 * np.log10(np.maximum(pg2, 1e-300)), np.nan).astype(np.float32)
                np.savez_compressed(cdir / f"noise_{sid}_{RT_FP}.npz", pl_db=pl2)
        print(f"band {band_tag(f)}: {len(SCENES)} scenes done in {time.time() - t_band:.1f}s")
    idx = pd.DataFrame(rows)
    idx.to_csv(ROOT / "rt" / f"index_{RT_FP}.csv", index=False)
    return idx

if CFG["RUN_RT"]:
    RT_INDEX = rt_phase()
else:
    RT_INDEX = pd.read_csv(ROOT / "rt" / f"index_{RT_FP}.csv")
print(RT_INDEX.groupby(["band", "split"]).agg(maps=("tx_idx", "count"), included=("include", "sum"),
      unhit=("invalid_frac", "mean"), at_floor=("floor_frac", "mean"), rt_s_median=("rt_s", "median")))
print(f"label policy: {LABEL_TAG}")


## 4. Phase 3 — The shared feature stack
Seven scene channels plus ten transmitter channels, computed once and cached as float16. Every model reads exactly these arrays.

In [ ]:
# ============================================================
# Cell 5 — Phase 3: the ONE shared feature stack (17 channels) used by every model
# ============================================================
SCENE_CH = ["height", "bmask", "bfrac10", "bfrac25", "bfrac50", "dist_bldg", "canyon"]   # Tx-independent
TX_CH = ["tx_h", "logd2", "logd3", "elev", "los", "ncross", "depth", "hmax_path", "h_exc", "nu"]  # per Tx
ALL_CH = SCENE_CH + TX_CH
FEATURE_SETS = {"B": ALL_CH, "A": ["height", "bmask", "tx_h", "logd2", "logd3", "elev", "los"]}
FEATS = FEATURE_SETS[CFG["FEATURE_SET"]]
FEAT_IDX = [ALL_CH.index(c) for c in FEATS]
FEATS_HASH = fingerprint(FEATS, 8)

def scene_features(H):
    bm = H > 0
    out = [H.astype(np.float32), bm.astype(np.float32)]
    for r in CFG["CONTEXT_RADII_M"]:
        rr = max(1, int(round(r / C)))
        yy, xx = np.mgrid[-rr:rr + 1, -rr:rr + 1]
        k = (xx ** 2 + yy ** 2 <= rr ** 2).astype(np.float32); k /= k.sum()
        out.append(np.clip(signal.fftconvolve(bm.astype(np.float32), k, mode="same"), 0, 1))
    dist = ndimage.distance_transform_edt(~bm) * C if bm.any() else np.full(H.shape, T, np.float32)
    out.append(np.minimum(dist, T).astype(np.float32))
    w = max(1, int(round(CFG["CANYON_HALF_WIDTH_M"] / C)))
    mf = ndimage.maximum_filter1d
    left, right = mf(bm, w, axis=1, origin=(w - 1) // 2), mf(bm, w, axis=1, origin=-(w // 2))
    down, up = mf(bm, w, axis=0, origin=(w - 1) // 2), mf(bm, w, axis=0, origin=-(w // 2))
    out.append((((left & right) | (down & up)) & ~bm).astype(np.float32))
    return np.stack(out)                                   # [7, N, N]

@torch.no_grad()
def link_features(H, tx_xyz, H_fine=None, chunk=4096, dev=DEVICE):
    """Vectorised ray walk Tx -> every pixel centre (all pixels at once). Obstructions are looked up in
    the finer raster H_fine (RAY_RASTER_UPSAMPLE x) when available; outputs live on the radio-map grid."""
    n = H.shape[0]
    Hl = H if H_fine is None else H_fine
    nf = Hl.shape[0]; cf = T / nf
    Ht = torch.as_tensor(Hl, dtype=torch.float32, device=dev).reshape(-1)
    X, Y = grid_centers(n)
    px = torch.as_tensor(np.stack([X.ravel(), Y.ravel()], 1), dtype=torch.float32, device=dev)
    tx = torch.as_tensor(tx_xyz[:2], dtype=torch.float32, device=dev)
    ztx, zrx = float(tx_xyz[2]), CFG["RX_H"]
    d2 = torch.linalg.norm(px - tx, dim=1)
    n_s = int(math.ceil(math.sqrt(2) * nf / CFG["RAY_STEP_PX"]))
    t = (torch.arange(n_s, device=dev, dtype=torch.float32) + 0.5) / n_s
    hl = ztx + t * (zrx - ztx)                             # LoS line height along the ray
    outs = {k: [] for k in ["los", "ncross", "depth", "hmax", "hexc", "d1"]}
    for s in range(0, px.shape[0], chunk):
        p = px[s:s + chunk]
        pts = tx[None, None, :] + t[None, :, None] * (p[:, None, :] - tx[None, None, :])
        j = ((pts[..., 0] + T / 2) / cf).floor().clamp(0, nf - 1).long()
        i = ((pts[..., 1] + T / 2) / cf).floor().clamp(0, nf - 1).long()
        hb = Ht[i * nf + j]
        exc = hb - hl[None, :]
        blk = exc > 0
        outs["los"].append(~blk.any(1))
        outs["ncross"].append((blk[:, 1:] & ~blk[:, :-1]).sum(1) + blk[:, 0].long())
        outs["depth"].append(blk.sum(1) * d2[s:s + chunk] / n_s)
        outs["hmax"].append(hb.max(1).values)
        mx, am = exc.max(1)
        outs["hexc"].append(mx)
        outs["d1"].append(t[am] * d2[s:s + chunk])
    o = {k: torch.cat(v).float().cpu().numpy().reshape(n, n) for k, v in outs.items()}
    d2n = d2.cpu().numpy().reshape(n, n)
    o["d2edge"] = np.maximum(d2n - o["d1"], 1.0)
    o["d1"] = np.maximum(o["d1"], 1.0)
    o["d2"] = d2n
    o["d3"] = np.sqrt(d2n ** 2 + (ztx - zrx) ** 2)
    o["elev"] = np.degrees(np.arctan2(ztx - zrx, np.maximum(d2n, 1e-3)))
    o["tx_h"] = np.full((n, n), ztx, np.float32)
    return o

def knife_edge_nu(lf, f_ghz):
    lam = 3e8 / (f_ghz * 1e9)
    nu = lf["hexc"] * np.sqrt(2 * (lf["d1"] + lf["d2edge"]) / (lam * lf["d1"] * lf["d2edge"]))
    return np.clip(nu, -10, 30).astype(np.float32)

def tx_feature_stack(lf, f_ghz):
    return np.stack([lf["tx_h"], np.log10(np.maximum(lf["d2"], 1.0)), np.log10(lf["d3"]), lf["elev"],
                     lf["los"].astype(np.float32), lf["ncross"], lf["depth"], lf["hmax"], lf["hexc"],
                     knife_edge_nu(lf, f_ghz)]).astype(np.float32)   # [10, N, N]

def scene_feat_path(sid):
    return ROOT / "feats" / f"scene_{sid}_{FEAT_FP}.npy"

def tx_feat_path(sid, f):
    return ROOT / "feats" / band_tag(f) / f"tx_{sid}_{FEAT_FP}.npy"

def feature_phase():
    timing = []
    for f in CFG["BANDS_GHZ"]:
        (ROOT / "feats" / band_tag(f)).mkdir(exist_ok=True)
    for _, m in SCENES.iterrows():
        sid = m["scene_id"]
        _sz = np.load(Path(m["dir"]) / "scene.npz")
        H, Hf = _sz["H"], (_sz["H_fine"] if "H_fine" in _sz.files else None)
        t0 = time.time()
        if not scene_feat_path(sid).exists():
            np.save(scene_feat_path(sid), scene_features(H).astype(np.float16))
        t_scene = time.time() - t0
        todo = [f for f in CFG["BANDS_GHZ"] if not tx_feat_path(sid, f).exists()]
        if todo:
            t1 = time.time()
            lfs = [link_features(H, np.array(xyz), Hf) for xyz in m["tx"]]
            t_link = (time.time() - t1) / len(lfs)
            for f in todo:
                np.save(tx_feat_path(sid, f), np.stack([tx_feature_stack(lf, f) for lf in lfs]).astype(np.float16))
            timing.append(dict(scene_id=sid, scene_s=t_scene, link_s_per_tx=t_link, n_tx=len(lfs)))
    tpath = ROOT / "feats" / f"timing_{FEAT_FP}.csv"
    if timing:
        old = pd.read_csv(tpath) if tpath.exists() else pd.DataFrame()
        pd.concat([old, pd.DataFrame(timing)]).drop_duplicates("scene_id", keep="last").to_csv(tpath, index=False)
    return pd.read_csv(tpath)

FEAT_TIMING = feature_phase() if CFG["RUN_FEATURES"] else pd.read_csv(ROOT / "feats" / f"timing_{FEAT_FP}.csv")
FEAT_S_PER_MAP = float((FEAT_TIMING["scene_s"] / FEAT_TIMING["n_tx"] + FEAT_TIMING["link_s_per_tx"]).median())
print(f"{len(ALL_CH)} channels: {ALL_CH}\nfeature set '{CFG['FEATURE_SET']}' -> {len(FEATS)} channels, "
      f"hash {FEATS_HASH}; median feature time per map = {FEAT_S_PER_MAP:.2f}s on {DEVICE}")


## 5. Splits, nested fractions, normalisation, fixed pixel sets

In [ ]:
# ============================================================
# Cell 6 — Dataset: city-level splits, nested fractions, shared normalisation, fixed pixel sets
# ============================================================
from functools import lru_cache

def localize():
    """Training reads every feature map many times: on Colab, copy features + labels from Drive to the
    local SSD once (Drive FUSE is far too slow for epoch-level reads). Returns the data root to read from."""
    if not (IN_COLAB or os.environ.get("RM_FORCE_LOCAL_CACHE") == "1"):
        return ROOT
    t0, n = time.time(), 0
    for sub in ["feats", "rt"]:
        for src in (ROOT / sub).rglob("*"):
            if src.is_file() and src.suffix in (".npy", ".npz"):
                dst = LOCAL_CACHE / src.relative_to(ROOT)
                if not dst.exists() or dst.stat().st_size != src.stat().st_size:
                    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
    print(f"local cache: copied {n} files to {LOCAL_CACHE} in {time.time() - t0:.0f}s")
    return LOCAL_CACHE

DATA_ROOT = localize()
def scene_feat_path(sid):
    return DATA_ROOT / "feats" / f"scene_{sid}_{FEAT_FP}.npy"
def tx_feat_path(sid, f):
    return DATA_ROOT / "feats" / band_tag(f) / f"tx_{sid}_{FEAT_FP}.npy"
RT_INDEX["rt_file"] = RT_INDEX["rt_file"].astype(str).str.replace(str(ROOT), str(DATA_ROOT), regex=False)

def maps_for_band(f):
    inc = (RT_INDEX["include"] if CFG["LABEL_MODE"] == "mask" else True)
    d = RT_INDEX[(RT_INDEX["band"] == f) & inc].copy()
    d["key"] = d["scene_id"] + "#" + d["tx_idx"].astype(str)
    return d.sort_values(["split", "scene_id", "tx_idx"]).reset_index(drop=True)

@lru_cache(maxsize=64)
def _scene_feat(sid):
    return np.load(scene_feat_path(sid), mmap_mode="r")

@lru_cache(maxsize=64)
def _tx_feat(sid, f):
    return np.load(tx_feat_path(sid, f), mmap_mode="r")

@lru_cache(maxsize=64)
def _rt(fn):
    """Labels after the label policy (identical for every model)."""
    d = np.load(fn)
    pl, valid, outdoor = d["pl_db"], d["valid"], d["outdoor"]
    if CFG["LABEL_MODE"] == "floor":
        pl = np.minimum(np.nan_to_num(pl, nan=CFG["PL_FLOOR_DB"]), CFG["PL_FLOOR_DB"]).astype(np.float32)
        valid = np.broadcast_to(outdoor, pl.shape).copy()
    return pl, valid

@lru_cache(maxsize=64)
def _rt_raw(fn):
    d = np.load(fn)
    return d["pl_db"], d["valid"]

def load_x(sid, k, f):
    """Feature stack [len(FEATS), N, N] float32 — identical for every model (raw, un-normalised)."""
    x = np.concatenate([_scene_feat(sid), _tx_feat(sid, f)[k]], 0)
    return np.asarray(x[FEAT_IDX], np.float32)

def load_y(row):
    pl, valid = _rt(row["rt_file"])
    return pl[row["tx_idx"]], valid[row["tx_idx"]]

def geo_los(sid, k, f):
    return np.asarray(_tx_feat(sid, f)[k][TX_CH.index("los")]) > 0.5

def train_scene_subset(train_scenes, frac, seed):
    """Nested by construction: one permutation per seed, prefix of length round(frac * n)."""
    perm = np.random.default_rng(1000 + seed).permutation(sorted(train_scenes))
    n = max(1, int(round(frac * len(perm))))
    return sorted(perm[:n].tolist())

def _key_rng(base, key):
    return np.random.default_rng(base + int(hashlib.md5(key.encode()).hexdigest()[:8], 16))

def fixed_pixels(maps, n_per_map, seed, tag, f):
    """Fixed valid-pixel indices per map, saved once and reused by every model (protocol P8)."""
    fn = ROOT / "rt" / band_tag(f) / f"pix_{tag}_{RT_FP}_{LABEL_TAG}_{seed}_{n_per_map}.pkl"
    if fn.exists():
        d = pickle.load(open(fn, "rb"))
        if set(maps["key"]) <= set(d):
            return d
    d = {}
    for _, r in maps.iterrows():
        _, v = load_y(r)
        idx = np.flatnonzero(v.ravel())
        rng = _key_rng(seed, r["key"])
        d[r["key"]] = np.sort(rng.choice(idx, min(n_per_map, len(idx)), replace=False))
    pickle.dump(d, open(fn, "wb"))
    return d

def pixset_hash(d):
    h = hashlib.sha1()
    for k in sorted(d):
        h.update(k.encode()); h.update(np.asarray(d[k]).tobytes())
    return h.hexdigest()[:10]

def norm_stats(maps, f, seed):
    """Per-channel mean/std from the TRAINING SUBSET only (all pixels), target stats from its valid pixels.
    The same statistics are applied to every model (protocol P1)."""
    xs, ys = [], []
    for _, r in maps.iterrows():
        x = load_x(r["scene_id"], r["tx_idx"], f).reshape(len(FEATS), -1)
        rng = _key_rng(seed + 77, r["key"])
        xs.append(x[:, rng.choice(x.shape[1], min(CFG["NORM_PIX_PER_MAP"], x.shape[1]), replace=False)])
        y, v = load_y(r)
        yv = y[v]
        ys.append(yv[rng.choice(len(yv), min(CFG["NORM_PIX_PER_MAP"], len(yv)), replace=False)])
    X = np.concatenate(xs, 1); Yc = np.concatenate(ys)
    mu, sd = X.mean(1), X.std(1)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return dict(mu=mu.astype(np.float32), sd=sd.astype(np.float32), ymu=float(Yc.mean()), ysd=float(Yc.std()))

def stats_hash(st):
    return fingerprint([np.round(st["mu"], 5).tolist(), np.round(st["sd"], 5).tolist(),
                        round(st["ymu"], 5), round(st["ysd"], 5)], 10)

def rows_at_pixels(maps, pix, st, f):
    """Tabular view of the shared feature stack at the given pixels (classical models)."""
    X, y, keys = [], [], []
    for _, r in maps.iterrows():
        idx = pix[r["key"]]
        x = load_x(r["scene_id"], r["tx_idx"], f).reshape(len(FEATS), -1)[:, idx].T
        yy, _ = load_y(r)
        X.append((x - st["mu"]) / st["sd"]); y.append((yy.ravel()[idx] - st["ymu"]) / st["ysd"])
        keys += [r["key"]] * len(idx)
    return np.concatenate(X).astype(np.float32), np.concatenate(y).astype(np.float32), np.array(keys)

def sample_train_pixels(maps, k, seed):
    """K valid pixels per training map, uniformly at random (natural LoS/NLoS mix)."""
    d = {}
    for _, r in maps.iterrows():
        _, v = load_y(r)
        idx = np.flatnonzero(v.ravel())
        d[r["key"]] = np.sort(_key_rng(seed + 31, r["key"]).choice(idx, min(k, len(idx)), replace=False))
    return d

class MapDataset(torch.utils.data.Dataset):
    """Image view of the shared feature stack (deep models)."""
    def __init__(self, maps, st, f, augment=False):
        self.m, self.st, self.f, self.aug = maps.reset_index(drop=True), st, f, augment
    def __len__(self):
        return len(self.m)
    def __getitem__(self, i):
        r = self.m.iloc[i]
        x = (load_x(r["scene_id"], r["tx_idx"], self.f) - self.st["mu"][:, None, None]) / self.st["sd"][:, None, None]
        y, v = load_y(r)
        y = np.where(v, (y - self.st["ymu"]) / self.st["ysd"], 0.0).astype(np.float32)
        if self.aug:                                      # D4 group (ablation only, protocol P9)
            kk, fl = np.random.randint(4), np.random.rand() < 0.5
            x, y, v = (np.rot90(a, kk, axes=(-2, -1)) for a in (x, y, v))
            if fl:
                x, y, v = (a[..., ::-1] for a in (x, y, v))
        return (torch.from_numpy(np.ascontiguousarray(x, np.float32)),
                torch.from_numpy(np.ascontiguousarray(y[None], np.float32)),
                torch.from_numpy(np.ascontiguousarray(v[None])))

def band_setup(f):
    maps = maps_for_band(f)
    sp = {s: maps[maps["split"] == s] for s in ["train", "val", "test"]}
    assert len(set(sp["train"]["city"]) & set(sp["test"]["city"])) == 0
    assert len(set(sp["train"]["city"]) & set(sp["val"]["city"])) == 0
    assert len(set(sp["val"]["city"]) & set(sp["test"]["city"])) == 0
    val_maps = sp["val"]
    if len(val_maps) > CFG["VAL_MAPS_MAX"]:
        val_maps = val_maps.sample(CFG["VAL_MAPS_MAX"], random_state=0).sort_values(["scene_id", "tx_idx"])
    eval_pix = fixed_pixels(sp["test"], CFG["EVAL_PIX_PER_MAP"], CFG["EVAL_PIX_SEED"], "test", f)
    val_pix = fixed_pixels(val_maps, CFG["K_PIX"], CFG["EVAL_PIX_SEED"] + 1, "val", f)
    return dict(f=f, maps=maps, train=sp["train"], val=val_maps, test=sp["test"],
                train_scenes=sorted(sp["train"]["scene_id"].unique()), eval_pix=eval_pix, val_pix=val_pix,
                eval_hash=pixset_hash({k: eval_pix[k] for k in sp["test"]["key"]}),
                n_train_maps_full=len(sp["train"]))

def subset_maps(B, frac, seed):
    sc = train_scene_subset(B["train_scenes"], frac, seed)
    return B["train"][B["train"]["scene_id"].isin(sc)].reset_index(drop=True)

BANDS = {f: band_setup(f) for f in CFG["BANDS_GHZ"]}
for f, B in BANDS.items():
    print(f"{band_tag(f)}: train maps={len(B['train'])} ({len(B['train_scenes'])} scenes), "
          f"val maps={len(B['val'])}, test maps={len(B['test'])}, eval-pixel hash={B['eval_hash']}")


## 6. Models


In [ ]:
# ============================================================
# Cell 7 — Models: classical learners, per-pixel MLP (1x1 conv), U-Nets and Swin-UNet
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

CLASSICAL = ["ridge", "knn", "svr", "rf", "lgbm"]
DEEP = ["mlp1x1", "unet_s", "unet_m", "unet_shallow", "swin_unet"]
FAMILY = dict(ridge="classical", knn="classical", svr="classical", rf="classical", lgbm="classical",
              mlp1x1="bridge", unet_s="deep", unet_m="deep", unet_shallow="deep", swin_unet="deep")
LABEL = dict(ridge="Ridge (poly)", knn="k-NN", svr="SVR", rf="RF", lgbm="LightGBM",
             mlp1x1="MLP", unet_s="UNet-S", unet_m="UNet-M", unet_shallow="UNet-shallow",
             swin_unet="Swin-UNet")

# ---------------- search spaces (trial 0 = sensible default) ----------------
def _lu(rng, a, b):
    return float(np.exp(rng.uniform(np.log(a), np.log(b))))

def sample_hp(model, rng, trial):
    if model == "ridge":
        d = dict(alpha=1.0, degree=2)
        return d if trial == 0 else dict(alpha=_lu(rng, 1e-4, 1e3), degree=int(rng.choice([1, 2, 3])))
    if model == "knn":
        d = dict(n_neighbors=15, weights="distance", p=2)
        return d if trial == 0 else dict(n_neighbors=int(round(_lu(rng, 3, 100))),
                                         weights=str(rng.choice(["uniform", "distance"])), p=int(rng.choice([1, 2])))
    if model == "svr":
        d = dict(C=10.0, gamma=0.1, epsilon=0.05)
        return d if trial == 0 else dict(C=_lu(rng, 0.1, 100), gamma=_lu(rng, 1e-3, 1.0), epsilon=_lu(rng, 0.01, 0.2))
    if model == "rf":
        d = dict(max_depth=None, min_samples_leaf=3, max_features=0.5)
        return d if trial == 0 else dict(max_depth=[None, 20, 30, 40][rng.integers(4)],
                                         min_samples_leaf=int(rng.choice([1, 3, 10, 30])),
                                         max_features=float(rng.choice([0.33, 0.5, 0.8, 1.0])))
    if model == "lgbm":
        d = dict(learning_rate=0.05, num_leaves=127, min_child_samples=50, subsample=0.8,
                 colsample_bytree=0.8, reg_lambda=1.0)
        return d if trial == 0 else dict(learning_rate=_lu(rng, 0.02, 0.2), num_leaves=int(rng.choice([31, 63, 127, 255])),
                                         min_child_samples=int(rng.choice([10, 50, 200])),
                                         subsample=float(rng.choice([0.6, 0.8, 1.0])),
                                         colsample_bytree=float(rng.choice([0.6, 0.8, 1.0])),
                                         reg_lambda=_lu(rng, 1e-3, 10))
    if model == "mlp1x1":
        d = dict(width=128, depth=3, lr=1e-3, wd=1e-4)
        return d if trial == 0 else dict(width=int(rng.choice([64, 128, 256])), depth=int(rng.choice([2, 3, 4])),
                                         lr=_lu(rng, 3e-4, 3e-3), wd=_lu(rng, 1e-5, 1e-2))
    if model in ("unet_s", "unet_m", "unet_shallow"):
        d = dict(lr=1e-3, wd=1e-4)
        return d if trial == 0 else dict(lr=_lu(rng, 2e-4, 2e-3), wd=_lu(rng, 1e-5, 1e-2))
    if model == "swin_unet":
        d = dict(lr=5e-4, wd=0.02, drop_path=0.1)
        return d if trial == 0 else dict(lr=_lu(rng, 1e-4, 1e-3), wd=_lu(rng, 1e-3, 5e-2),
                                         drop_path=float(rng.choice([0.0, 0.1, 0.2])))
    raise KeyError(model)

# ---------------- classical ----------------
def make_classical(model, hp, n_rows, seed):
    if model == "ridge":
        return make_pipeline(PolynomialFeatures(hp["degree"], include_bias=False), Ridge(alpha=hp["alpha"]))
    if model == "knn":
        return KNeighborsRegressor(n_neighbors=hp["n_neighbors"], weights=hp["weights"], p=hp["p"], n_jobs=-1)
    if model == "svr":
        return SVR(kernel="rbf", C=hp["C"], gamma=hp["gamma"], epsilon=hp["epsilon"], cache_size=2000)
    if model == "rf":
        ms = min(1.0, CFG["RF_MAX_SAMPLES"] / max(1, n_rows))
        return RandomForestRegressor(n_estimators=CFG["RF_N_EST"], max_depth=hp["max_depth"],
                                     min_samples_leaf=hp["min_samples_leaf"], max_features=hp["max_features"],
                                     max_samples=ms if ms < 1.0 else None, n_jobs=-1, random_state=seed)
    if model == "lgbm":
        return lgb.LGBMRegressor(n_estimators=5000, subsample_freq=1, random_state=seed, n_jobs=-1, verbose=-1, **hp)
    raise KeyError(model)

def classical_cap(model):
    return dict(knn=CFG["KNN_CAP"], svr=CFG["SVR_CAP"]).get(model)

def classical_complexity(model, est):
    """Model 'size' on one axis: stored floats / coefficients / tree nodes / support vectors."""
    if model == "ridge":
        return dict(n_params=int(est[-1].coef_.size + 1))
    if model == "knn":
        return dict(n_params=int(est._fit_X.size))
    if model == "svr":
        return dict(n_params=int(est.support_vectors_.size + est.dual_coef_.size), n_support=int(len(est.support_)))
    if model == "rf":
        return dict(n_params=int(sum(t.tree_.node_count for t in est.estimators_)))
    if model == "lgbm":
        dm = est.booster_.dump_model()
        return dict(n_params=int(sum(ti["num_leaves"] * 2 - 1 for ti in dm["tree_info"])),
                    n_trees=int(est.booster_.num_trees()))
    return {}

# ---------------- deep ----------------
class MLP1x1(nn.Module):
    """Per-pixel MLP == network of 1x1 convolutions: same learner class as an MLP, zero spatial context."""
    def __init__(self, cin, width=128, depth=3):
        super().__init__()
        L, c = [], cin
        for _ in range(depth):
            L += [nn.Conv2d(c, width, 1), nn.GELU()]; c = width
        L += [nn.Conv2d(c, 1, 1)]
        self.net = nn.Sequential(*L)
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Sequential):
    def __init__(self, a, b):
        super().__init__(nn.Conv2d(a, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
                         nn.Conv2d(b, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, cin, base=32, levels=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(levels + 1)]
        self.inc = DoubleConv(cin, ch[0])
        self.down = nn.ModuleList([DoubleConv(ch[i], ch[i + 1]) for i in range(levels)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(levels))])
        self.dec = nn.ModuleList([DoubleConv(ch[i] * 2, ch[i]) for i in reversed(range(levels))])
        self.out = nn.Conv2d(ch[0], 1, 1)
    def forward(self, x):
        s = [self.inc(x)]
        for d in self.down:
            s.append(d(F.max_pool2d(s[-1], 2)))
        h = s.pop()
        for up, dec in zip(self.up, self.dec):
            h = dec(torch.cat([up(h), s.pop()], 1))
        return self.out(h)

# --- compact Swin-UNet (Cao et al., 2021 style): window attention + shifted windows, patch merge/expand ---
def _win_part(x, w):            # x: B,H,W,C -> (B*nW), w*w, C
    B, H, W, C_ = x.shape
    x = x.view(B, H // w, w, W // w, w, C_).permute(0, 1, 3, 2, 4, 5)
    return x.reshape(-1, w * w, C_)

def _win_rev(xw, w, B, H, W):
    C_ = xw.shape[-1]
    x = xw.view(B, H // w, W // w, w, w, C_).permute(0, 1, 3, 2, 4, 5)
    return x.reshape(B, H, W, C_)

class WindowAttention(nn.Module):
    def __init__(self, dim, heads, w):
        super().__init__()
        self.h, self.w, self.scale = heads, w, (dim // heads) ** -0.5
        self.qkv, self.proj = nn.Linear(dim, dim * 3), nn.Linear(dim, dim)
        self.bias = nn.Parameter(torch.zeros((2 * w - 1) ** 2, heads))
        nn.init.trunc_normal_(self.bias, std=0.02)
        c = torch.stack(torch.meshgrid(torch.arange(w), torch.arange(w), indexing="ij")).flatten(1)
        rel = (c[:, :, None] - c[:, None, :]).permute(1, 2, 0) + (w - 1)
        self.register_buffer("rel_idx", rel[..., 0] * (2 * w - 1) + rel[..., 1], persistent=False)
    def forward(self, x, mask=None):
        Bn, N_, C_ = x.shape
        q, k, v = self.qkv(x).reshape(Bn, N_, 3, self.h, C_ // self.h).permute(2, 0, 3, 1, 4)
        a = (q * self.scale) @ k.transpose(-2, -1)
        a = a + self.bias[self.rel_idx.reshape(-1)].reshape(N_, N_, -1).permute(2, 0, 1)[None]
        if mask is not None:
            nW = mask.shape[0]
            a = a.view(Bn // nW, nW, self.h, N_, N_) + mask[None, :, None]
            a = a.view(-1, self.h, N_, N_)
        return self.proj((a.softmax(-1) @ v).transpose(1, 2).reshape(Bn, N_, C_))

class SwinBlock(nn.Module):
    def __init__(self, dim, heads, res, w, shift, drop_path=0.0):
        super().__init__()
        self.res = res
        self.w = min(w, res)
        self.shift = 0 if res <= w else shift
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, heads, self.w)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.dp = drop_path
        if self.shift:
            img = torch.zeros(1, res, res, 1)
            cnt = 0
            for hs in (slice(0, -self.w), slice(-self.w, -self.shift), slice(-self.shift, None)):
                for ws in (slice(0, -self.w), slice(-self.w, -self.shift), slice(-self.shift, None)):
                    img[:, hs, ws, :] = cnt; cnt += 1
            mw = _win_part(img, self.w).squeeze(-1)
            m = mw[:, None, :] - mw[:, :, None]
            self.register_buffer("mask", m.masked_fill(m != 0, -100.0).masked_fill(m == 0, 0.0), persistent=False)
        else:
            self.mask = None
    def _drop(self, x):
        if not self.training or self.dp == 0:
            return x
        keep = (torch.rand(x.shape[0], 1, 1, device=x.device) >= self.dp).to(x.dtype)
        return x * keep / (1 - self.dp)
    def forward(self, x):                      # x: B, L, C
        B, L, C_ = x.shape
        H = W = self.res
        h = self.n1(x).view(B, H, W, C_)
        if self.shift:
            h = torch.roll(h, (-self.shift, -self.shift), (1, 2))
        hw = self.attn(_win_part(h, self.w), self.mask)
        h = _win_rev(hw, self.w, B, H, W)
        if self.shift:
            h = torch.roll(h, (self.shift, self.shift), (1, 2))
        x = x + self._drop(h.reshape(B, L, C_))
        return x + self._drop(self.mlp(self.n2(x)))

class PatchMerge(nn.Module):
    def __init__(self, dim, res):
        super().__init__(); self.res = res
        self.norm, self.red = nn.LayerNorm(4 * dim), nn.Linear(4 * dim, 2 * dim, bias=False)
    def forward(self, x):
        B, L, C_ = x.shape; H = self.res
        x = x.view(B, H, H, C_)
        x = torch.cat([x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]], -1)
        return self.red(self.norm(x.view(B, -1, 4 * C_)))

class PatchExpand(nn.Module):
    def __init__(self, dim, res, scale=2, out_dim=None):
        super().__init__(); self.res, self.s = res, scale
        out_dim = out_dim or dim // 2
        self.lin = nn.Linear(dim, out_dim * scale * scale, bias=False)
        self.norm, self.od = nn.LayerNorm(out_dim), out_dim
    def forward(self, x):
        B, L, _ = x.shape; H, s, od = self.res, self.s, self.od
        x = self.lin(x).view(B, H, H, s, s, od).permute(0, 1, 3, 2, 4, 5).reshape(B, H * s * H * s, od)
        return self.norm(x)

class SwinUNet(nn.Module):
    def __init__(self, cin, img, patch=4, dim=64, depths=(2, 2, 2, 2), heads=(2, 4, 8, 16), window=8, drop_path=0.1):
        super().__init__()
        self.r0, nS = img // patch, len(depths)
        self.embed = nn.Conv2d(cin, dim, patch, stride=patch)
        self.enorm = nn.LayerNorm(dim)
        dps = np.linspace(0, drop_path, sum(depths)).tolist()
        self.enc, self.merge, self.dec, self.expand, self.fuse = (nn.ModuleList() for _ in range(5))
        k = 0
        for s in range(nS):
            d, r = dim * 2 ** s, self.r0 // 2 ** s
            self.enc.append(nn.Sequential(*[SwinBlock(d, heads[s], r, window, window // 2 if b % 2 else 0, dps[k + b])
                                            for b in range(depths[s])]))
            k += depths[s]
            if s < nS - 1:
                self.merge.append(PatchMerge(d, r))
        for s in reversed(range(nS - 1)):
            d, r = dim * 2 ** s, self.r0 // 2 ** s
            self.expand.append(PatchExpand(d * 2, r // 2))
            self.fuse.append(nn.Linear(2 * d, d))
            self.dec.append(nn.Sequential(*[SwinBlock(d, heads[s], r, window, window // 2 if b % 2 else 0)
                                            for b in range(depths[s])]))
        self.final = PatchExpand(dim, self.r0, scale=patch, out_dim=dim)
        self.head = nn.Conv2d(dim, 1, 1)
        self.patch, self.dim = patch, dim
    def forward(self, x):
        B = x.shape[0]
        h = self.enorm(self.embed(x).flatten(2).transpose(1, 2))
        skips = []
        for s, blk in enumerate(self.enc):
            h = blk(h)
            if s < len(self.merge):
                skips.append(h); h = self.merge[s](h)
        for exp, fuse, blk in zip(self.expand, self.fuse, self.dec):
            h = blk(fuse(torch.cat([exp(h), skips.pop()], -1)))
        h = self.final(h)
        H = self.r0 * self.patch
        return self.head(h.view(B, H, H, self.dim).permute(0, 3, 1, 2))

def make_deep(model, hp, cin):
    if model == "mlp1x1":
        return MLP1x1(cin, hp["width"], hp["depth"])
    if model == "unet_s":
        return UNet(cin, base=12, levels=4)
    if model == "unet_m":
        return UNet(cin, base=36, levels=4)
    if model == "unet_shallow":      # same width as UNet-M, receptive field ~ 1/4 (2 levels)
        return UNet(cin, base=36, levels=2)
    if model == "swin_unet":
        return SwinUNet(cin, N_PIX, patch=4, dim=64, depths=(2, 2, 2, 2), heads=(2, 4, 8, 16), window=8,
                        drop_path=hp.get("drop_path", 0.1))
    raise KeyError(model)

def count_params(m):
    return int(sum(p.numel() for p in m.parameters()))

def flops_per_map(m, cin):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        m.eval()
        x = torch.zeros(1, cin, N_PIX, N_PIX, device=next(m.parameters()).device)
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            m(x)
        return int(fc.get_total_flops())
    except Exception as e:
        print("flop count failed:", e)
        return None

for _m in ["mlp1x1", "unet_s", "unet_m", "unet_shallow", "swin_unet"]:
    _net = make_deep(_m, sample_hp(_m, np.random.default_rng(0), 0), len(FEATS))
    _y = _net(torch.zeros(1, len(FEATS), N_PIX, N_PIX))
    assert _y.shape == (1, 1, N_PIX, N_PIX), (_m, _y.shape)
    print(f"{LABEL[_m]:>15s}: {count_params(_net) / 1e6:6.2f} M params")
del _net, _y


## 7. Training, evaluation and cost accounting

In [ ]:
# ============================================================
# Cell 8 — Training / prediction / evaluation core with cost accounting
# ============================================================
class PeakRSS:
    """Samples process RSS in a background thread; .peak_mb = peak above the starting level."""
    def __enter__(self):
        self.p = psutil.Process(); self.base = self.p.memory_info().rss; self.peak = self.base
        self.stop = False
        def run():
            while not self.stop:
                self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(0.05)
        self.t = threading.Thread(target=run, daemon=True); self.t.start()
        return self
    def __exit__(self, *a):
        self.stop = True; self.t.join()
        self.peak_mb = (self.peak - self.base) / 2 ** 20

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def _rmse(pred_db, true_db):
    return float(np.sqrt(np.mean((pred_db - true_db) ** 2)))

# ---------------- classical ----------------
def fit_classical(model, hp, B, tr, st, seed, k_pix=None):
    f = B["f"]
    pix = sample_train_pixels(tr, k_pix or CFG["K_PIX"], seed)
    X, y, _ = rows_at_pixels(tr, pix, st, f)
    n_avail, cap = len(y), classical_cap(model)
    if cap and n_avail > cap:
        sel = np.random.default_rng(seed + 5).choice(n_avail, cap, replace=False)
        X, y = X[sel], y[sel]
    est = make_classical(model, hp, len(y), seed)
    if model == "lgbm":
        Xv, yv, _ = rows_at_pixels(B["val"], B["val_pix"], st, f)
        est.fit(X, y, eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(100, verbose=False)])
    else:
        est.fit(X, y)
    return est, dict(rows_available=int(n_avail), rows_used=int(len(y)), cap_bound=bool(cap and n_avail > cap))

def predict_classical(est, X, chunk=262_144):
    return np.concatenate([est.predict(X[i:i + chunk]) for i in range(0, len(X), chunk)]) if len(X) else np.zeros(0)

def classical_full_map(est, r, st, f):
    x = load_x(r["scene_id"], r["tx_idx"], f).reshape(len(FEATS), -1).T
    return predict_classical(est, (x - st["mu"]) / st["sd"]).reshape(N_PIX, N_PIX) * st["ysd"] + st["ymu"]

# ---------------- deep ----------------
def _amp_ctx():
    if CFG["AMP"] and DEVICE.type == "cuda":
        dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.autocast("cuda", dtype=dt)
    import contextlib
    return contextlib.nullcontext()

@torch.no_grad()
def deep_predict_maps(net, maps, st, f, bs=None):
    """Yields (row, predicted path-loss map in dB)."""
    net.eval()
    ds = MapDataset(maps, st, f)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs or CFG["BATCH"], shuffle=False, num_workers=CFG["NUM_WORKERS"])
    i = 0
    for x, _, _ in dl:
        with _amp_ctx():
            p = net(x.to(DEVICE)).float().cpu().numpy()[:, 0]
        for k in range(len(p)):
            yield maps.iloc[i], p[k] * st["ysd"] + st["ymu"]; i += 1

def val_rmse_pixels(pred_iter, pixset):
    se, n = 0.0, 0
    for r, pdb in pred_iter:
        idx = pixset[r["key"]]
        y, _ = load_y(r)
        d = pdb.ravel()[idx] - y.ravel()[idx]
        se += float((d ** 2).sum()); n += len(idx)
    return math.sqrt(se / max(1, n))

def fit_deep(model, hp, B, tr, st, seed, augment=False):
    f = B["f"]
    set_seed(seed)
    net = make_deep(model, hp, len(FEATS)).to(DEVICE)
    ds = MapDataset(tr, st, f, augment=augment)
    bs = min(CFG["BATCH"], len(ds))
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True,
                                     num_workers=CFG["NUM_WORKERS"], persistent_workers=CFG["NUM_WORKERS"] > 0,
                                     generator=torch.Generator().manual_seed(seed))
    steps = int(np.clip(CFG["EPOCHS_EQ"] * len(ds) / bs, CFG["MIN_STEPS"], CFG["MAX_STEPS"]))
    ev = CFG["EVAL_EVERY"] or max(50, steps // 30)
    opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    warm = max(1, steps // 20)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    use_scaler = CFG["AMP"] and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    best, best_state, bad, step = float("inf"), None, 0, 0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    while step < steps and bad < CFG["PATIENCE"]:
        for x, y, v in dl:
            net.train()
            x, y, v = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), v.to(DEVICE, non_blocking=True)
            with _amp_ctx():
                p = net(x)
            loss = (((p.float() - y) ** 2) * v).sum() / v.sum().clamp(min=1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if step % ev == 0 or step == steps:
                vr = val_rmse_pixels(deep_predict_maps(net, B["val"], st, f), B["val_pix"])
                if vr < best - 1e-4:
                    best, bad = vr, 0
                    best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
                else:
                    bad += 1
            if step >= steps or bad >= CFG["PATIENCE"]:
                break
    fit_s = time.time() - t0
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE.type == "cuda" else None
    return net, dict(steps=int(step), steps_budget=steps, best_val=float(best), fit_s=fit_s, gpu_peak_mb=peak)

# ---------------- evaluation on the fixed test pixels (+ optional full maps) ----------------
def permap_stats(pred_px, true_px, los_px):
    e = pred_px - true_px
    cov = true_px < CFG["PL_FLOOR_DB"] - 1e-3          # in-coverage pixels (label not clipped)
    return dict(n=len(e), sse=float((e ** 2).sum()), sae=float(np.abs(e).sum()), se=float(e.sum()),
                n_los=int(los_px.sum()), sse_los=float((e[los_px] ** 2).sum()),
                n_nlos=int((~los_px).sum()), sse_nlos=float((e[~los_px] ** 2).sum()),
                n_cov=int(cov.sum()), sse_cov=float((e[cov] ** 2).sum()))

def evaluate(model, est_or_net, B, st, full):
    f, test = B["f"], B["test"]
    rows, full_se, full_n = [], 0.0, 0
    if FAMILY[model] == "classical":
        X, _, keys = rows_at_pixels(test, B["eval_pix"], st, f)
        pred = predict_classical(est_or_net, X) * st["ysd"] + st["ymu"]
        off = 0
        for _, r in test.iterrows():
            idx = B["eval_pix"][r["key"]]; n = len(idx)
            y, v = load_y(r)
            los = geo_los(r["scene_id"], r["tx_idx"], f).ravel()[idx]
            rows.append(dict(key=r["key"], **permap_stats(pred[off:off + n], y.ravel()[idx], los))); off += n
            if full:
                pm = classical_full_map(est_or_net, r, st, f)
                full_se += float(((pm - y)[v] ** 2).sum()); full_n += int(v.sum())
    else:
        for r, pdb in deep_predict_maps(est_or_net, test, st, f):
            idx = B["eval_pix"][r["key"]]
            y, v = load_y(r)
            los = geo_los(r["scene_id"], r["tx_idx"], f).ravel()[idx]
            rows.append(dict(key=r["key"], **permap_stats(pdb.ravel()[idx], y.ravel()[idx], los)))
            if full:
                full_se += float(((pdb - y)[v] ** 2).sum()); full_n += int(v.sum())
    pm = pd.DataFrame(rows)
    agg = dict(rmse=math.sqrt(pm.sse.sum() / pm.n.sum()), mae=pm.sae.sum() / pm.n.sum(), bias=pm.se.sum() / pm.n.sum(),
               rmse_los=math.sqrt(pm.sse_los.sum() / max(1, pm.n_los.sum())),
               rmse_nlos=math.sqrt(pm.sse_nlos.sum() / max(1, pm.n_nlos.sum())),
               rmse_cov=math.sqrt(pm.sse_cov.sum() / max(1, pm.n_cov.sum())),
               rmse_full=(math.sqrt(full_se / full_n) if full and full_n else None))
    return agg, pm

def inference_ms(model, est_or_net, B, st, n_rep=3):
    """Wall time to predict ONE full map from its (pre-computed, shared) feature stack."""
    f, test = B["f"], B["test"]
    rs = [test.iloc[i % len(test)] for i in range(n_rep)]
    if FAMILY[model] == "classical":
        ts = []
        for r in rs:
            t0 = time.perf_counter(); classical_full_map(est_or_net, r, st, f); ts.append(time.perf_counter() - t0)
        return 1000 * float(np.median(ts))
    net = est_or_net.eval()
    x = MapDataset(test.iloc[:1], st, f)[0][0][None].to(DEVICE)
    ts = []
    with torch.no_grad():
        for i in range(n_rep + 2):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with _amp_ctx():
                net(x)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            if i >= 2:
                ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts))

def size_mb(model, est_or_net):
    if FAMILY[model] == "classical":
        if model == "lgbm":
            return len(est_or_net.booster_.model_to_string().encode()) / 2 ** 20
        return len(pickle.dumps(est_or_net)) / 2 ** 20
    return sum(t.numel() * t.element_size() for t in est_or_net.state_dict().values()) / 2 ** 20

def train_once(model, hp, B, tr, st, seed, augment=False, k_pix=None):
    """One fit with cost accounting. Returns (est/net, info)."""
    with PeakRSS() as mem:
        t0 = time.time()
        if FAMILY[model] == "classical":
            est, info = fit_classical(model, hp, B, tr, st, seed, k_pix=k_pix)
            info["fit_s"] = time.time() - t0
            info.update(classical_complexity(model, est))
        else:
            est, info = fit_deep(model, hp, B, tr, st, seed, augment=augment)
            info["n_params"] = count_params(est)
    info["cpu_peak_mb"] = mem.peak_mb
    return est, info

def val_score(model, est, B, st):
    if FAMILY[model] == "classical":
        Xv, yv, _ = rows_at_pixels(B["val"], B["val_pix"], st, B["f"])
        p = predict_classical(est, Xv)
        return float(np.sqrt(np.mean((p - yv) ** 2))) * st["ysd"]
    return val_rmse_pixels(deep_predict_maps(est, B["val"], st, B["f"]), B["val_pix"])


## 8. Phase 4 — Tuning and the experiment grid

In [ ]:
# ============================================================
# Cell 9 — Phase 4: equal-budget tuning, then the full fraction x seed grid (resumable)
# ============================================================
RESULTS_CSV, PERMAP_CSV, TUNE_CSV = EXP_DIR / "results.jsonl", EXP_DIR / "permap.csv", EXP_DIR / "tuning_log.csv"
ALL_MODELS = CFG["MODELS"] + [m for m in CFG["OPTIONAL_MODELS"] if m not in CFG["MODELS"]]

def _append(path, rows):
    rows = rows if isinstance(rows, list) else [rows]
    if path.suffix == ".jsonl":                      # schema-free: classical and deep rows carry different fields
        with open(path, "a") as fh:
            for r in rows:
                fh.write(json.dumps({k: (None if isinstance(v, float) and not np.isfinite(v) else
                                         (v.item() if hasattr(v, "item") else v)) for k, v in r.items()}) + "\n")
    else:
        pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)

def read_results():
    """Results log. precise_float keeps fractions exact (0.3 stays 0.3), so equality filters work."""
    if not RESULTS_CSV.exists():
        return pd.DataFrame()
    d = pd.read_json(RESULTS_CSV, lines=True, precise_float=True)
    d["frac"] = d["frac"].round(6)
    return d

def seeds_for(frac):
    return CFG["SEEDS_SMALL"] if frac <= CFG["SMALL_FRAC_MAX"] + 1e-9 else CFG["SEEDS_LARGE"]

def _trial_log(model, f):
    return EXP_DIR / "tuning_trials" / f"{band_tag(f)}_{model}.jsonl"

def tune(model, B):
    """Random search with the SAME number of trials for every model (protocol P7), at the tuning
    fraction, selected on the fixed validation pixels (validation cities). Settings then frozen.
    Resumable per TRIAL: every finished trial is appended to tuning_trials/<band>_<model>.jsonl on Drive;
    after an interruption the RNG is replayed so the remaining trials draw exactly the same configs."""
    fn = EXP_DIR / f"hp_{band_tag(B['f'])}_{model}.json"
    if fn.exists():
        return json.load(open(fn))
    tl = _trial_log(model, B["f"]); tl.parent.mkdir(exist_ok=True)
    done = pd.read_json(tl, lines=True).to_dict("records") if tl.exists() else []
    tr = subset_maps(B, CFG["TUNE_FRACTION"], CFG["TUNE_SEED"])
    st = norm_stats(tr, B["f"], CFG["TUNE_SEED"])
    rng = np.random.default_rng(int(hashlib.md5(model.encode()).hexdigest()[:8], 16))
    if done:
        print(f"   resuming {LABEL[model]} tuning at trial {len(done)}/{CFG['TUNE_TRIALS']}")
    for trial in range(CFG["TUNE_TRIALS"]):
        hp = sample_hp(model, rng, trial)                  # always drawn, so the RNG stream is identical
        if trial < len(done):
            if json.loads(done[trial]["hp"]) != json.loads(json.dumps(hp)):
                print(f"   WARNING: replayed hp for trial {trial} differs from the log; using the logged result")
            continue
        t0, est = time.time(), None
        try:
            est, info = train_once(model, hp, B, tr, st, CFG["TUNE_SEED"])
            vs = val_score(model, est, B, st)
        except Exception as e:                      # a failed config counts as a used trial
            print(f"   trial {trial} failed: {e}"); vs = float("inf")
        rec = dict(band=B["f"], model=model, trial=trial, hp=json.dumps(hp), val_rmse=vs,
                   seconds=time.time() - t0, n_train_maps=len(tr))
        _append(tl, rec); done.append(rec)
        print(f"   {LABEL[model]} trial {trial + 1}/{CFG['TUNE_TRIALS']}: val RMSE {vs:.2f} dB ({rec['seconds']:.0f}s)")
        del est
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    _v = lambda x: float("inf") if x is None or (isinstance(x, float) and not np.isfinite(x)) else float(x)
    best = min(done, key=lambda r: _v(r["val_rmse"]))
    out = dict(model=model, band=B["f"], hp=json.loads(best["hp"]), val_rmse=best["val_rmse"],
               trials=CFG["TUNE_TRIALS"], tune_s=float(sum(r["seconds"] for r in done)),   # summed over sessions
               tune_fraction=CFG["TUNE_FRACTION"], n_train_maps=len(tr))
    json.dump(out, open(fn, "w"), indent=1)
    _append(TUNE_CSV, done)
    print(f"   tuned {LABEL[model]}: val RMSE {out['val_rmse']:.2f} dB  hp={out['hp']}  ({out['tune_s']:.0f}s)")
    return out

def save_model(model, est, st, row):
    """Persist every trained model + the exact normalisation it needs (Drive), so predictions can be
    regenerated later (qualitative figures, further analysis) without retraining."""
    if not CFG.get("SAVE_MODELS", True):
        return None
    d = EXP_DIR / "models" / band_tag(row["band"]); d.mkdir(parents=True, exist_ok=True)
    stem = f"{model}_f{row['frac']}_s{row['seed']}_aug{int(row['augment'])}"
    meta = dict(model=model, hp=json.loads(row["hp"]), feats=FEATS, stats={k: (v.tolist() if hasattr(v, "tolist") else v)
                for k, v in st.items()}, run_key=row["run_key"], label_tag=LABEL_TAG, exp_fp=EXP_FP)
    if FAMILY[model] == "classical":
        path = d / f"{stem}.pkl"
        with open(path, "wb") as fh:
            pickle.dump(dict(est=est, **meta), fh, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        path = d / f"{stem}.pt"
        torch.save(dict(state_dict={k: v.cpu() for k, v in est.state_dict().items()}, **meta), path)
    return str(path)

def load_saved_model(path):
    """Rebuild a saved model: returns (estimator_or_network, meta)."""
    path = Path(path)
    if path.suffix == ".pkl":
        d = pickle.load(open(path, "rb"))
        return d.pop("est"), d
    d = torch.load(path, map_location=DEVICE, weights_only=False)
    net = make_deep(d["model"], d["hp"], len(d["feats"])).to(DEVICE)
    net.load_state_dict(d["state_dict"]); net.eval()
    return net, {k: v for k, v in d.items() if k != "state_dict"}

def done_keys():
    d = read_results()
    return set(d["run_key"]) if len(d) else set()

def run_one(model, B, frac, seed, hpd, augment=False):
    f = B["f"]
    tr = subset_maps(B, frac, seed)
    st = norm_stats(tr, f, seed)
    est, info = train_once(model, hpd["hp"], B, tr, st, seed, augment=augment)
    full = (FAMILY[model] != "classical") or (abs(frac - 1.0) < 1e-9 and model not in CFG["FULL_EVAL_SKIP"])
    agg, pm = evaluate(model, est, B, st, full)
    cin = len(FEATS)
    row = dict(run_key=f"{band_tag(f)}|{model}|{frac}|{seed}|aug{int(augment)}", exp_fp=EXP_FP, band=f, model=model,
               family=FAMILY[model], augment=augment, frac=frac, seed=seed,
               n_scenes=int(tr["scene_id"].nunique()), n_maps=len(tr), frac_actual=len(tr) / B["n_train_maps_full"],
               feats_hash=FEATS_HASH, stats_hash=stats_hash(st), eval_hash=B["eval_hash"],
               hp=json.dumps(hpd["hp"]), tune_s=hpd["tune_s"], tune_trials=hpd["trials"],
               rt_s_per_map=float(B["train"]["rt_s"].median()), feat_s_per_map=FEAT_S_PER_MAP,
               infer_ms=inference_ms(model, est, B, st), size_mb=size_mb(model, est),
               flops=(flops_per_map(est, cin) if FAMILY[model] != "classical" else None),
               timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), **agg, **info)
    row["model_path"] = save_model(model, est, st, row)       # saved BEFORE the row is logged
    _append(RESULTS_CSV, row)
    _append(PERMAP_CSV, pm.assign(run_key=row["run_key"]).to_dict("records"))
    del est
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return row

def experiment_phase():
    for f, B in BANDS.items():
        print(f"===== band {band_tag(f)} =====")
        hps = {m: tune(m, B) for m in ALL_MODELS}
        done = done_keys()
        grid = [(m, fr, s) for m in ALL_MODELS for fr in CFG["FRACTIONS"] for s in seeds_for(fr)]
        if CFG["RUN_AUG_ABLATION"]:
            aug_fr = sorted({CFG["FRACTIONS"][0], CFG["TUNE_FRACTION"], 1.0})
            grid_aug = [(m, fr, 0) for m in ALL_MODELS if FAMILY[m] != "classical" for fr in aug_fr]
        else:
            grid_aug = []
        todo = [(g, False) for g in grid] + [(g, True) for g in grid_aug]
        t0 = time.time()
        for n, ((m, fr, s), aug) in enumerate(todo):
            key = f"{band_tag(f)}|{m}|{fr}|{s}|aug{int(aug)}"
            if key in done:
                continue
            r = run_one(m, B, fr, s, hps[m], augment=aug)
            print(f"[{n + 1}/{len(todo)}] {LABEL[m]:>15s} f={fr:<5} seed={s} aug={int(aug)}: RMSE {r['rmse']:.2f} dB "
                  f"(LoS {r['rmse_los']:.2f} / NLoS {r['rmse_nlos']:.2f}), fit {r['fit_s']:.0f}s, "
                  f"elapsed {(time.time() - t0) / 60:.1f} min")

if CFG["RUN_EXPERIMENTS"]:
    experiment_phase()
RES = read_results()
RES["augment"] = RES["augment"].astype(bool)
PERMAP = pd.read_csv(PERMAP_CSV)
print(f"{len(RES)} runs logged in {RESULTS_CSV}")

## 9. Phase 5 — Analysis
### View 1: fixed data → compare accuracy

In [ ]:
# ============================================================
# Cell 10 — Phase 5a: analysis helpers + VIEW 1 (fixed data -> compare accuracy)
# ============================================================
ORDER = [m for m in ["ridge", "knn", "svr", "rf", "lgbm", "mlp1x1", "unet_shallow", "unet_s", "unet_m", "swin_unet"]
         if m in ALL_MODELS]
STYLE = {"classical": "-", "bridge": "-.", "deep": "--"}
COLORS = dict(zip(ORDER, plt.cm.tab10(np.linspace(0, 1, 10))))
alpha_ci = (1 - CFG["CI"]) / 2

def apply_label_policy(pl):
    if CFG["LABEL_MODE"] == "floor":
        return np.minimum(np.nan_to_num(pl, nan=CFG["PL_FLOOR_DB"]), CFG["PL_FLOOR_DB"])
    return pl

def label_noise_floor(band):
    """Monte-Carlo label noise: same scene and settings, two RT seeds, compared under the label policy.
    sigma = RMS(difference)/sqrt(2) is the irreducible RMSE of a perfect predictor of the expected map."""
    rows = []
    for fn in sorted((ROOT / "rt_checks" / band_tag(band)).glob(f"noise_*_{RT_FP}.npz")):
        sid = fn.stem[len("noise_"):-len(f"_{RT_FP}")]
        r = RT_INDEX[(RT_INDEX.scene_id == sid) & (RT_INDEX.band == band)].iloc[0]
        pl1, v1 = _rt(r["rt_file"])
        pl2 = apply_label_policy(np.load(fn)["pl_db"])
        for k in range(pl1.shape[0]):
            ok = v1[k] & np.isfinite(pl2[k])
            d = pl1[k][ok] - pl2[k][ok]
            rows.append(dict(scene_id=sid, tx=k, rmse_between_seeds=float(np.sqrt(np.mean(d ** 2))),
                             label_sigma=float(np.sqrt(np.mean(d ** 2)) / np.sqrt(2)),
                             median_abs_diff=float(np.median(np.abs(d))), p95_abs_diff=float(np.percentile(np.abs(d), 95))))
    return pd.DataFrame(rows)

NOISE = {f: label_noise_floor(f) for f in BANDS}
NOISE_SIGMA = {f: (float(np.sqrt((NOISE[f].label_sigma ** 2).mean())) if len(NOISE[f]) else 0.0) for f in BANDS}

def headline(band):
    return RES[(RES["band"] == band) & (~RES["augment"].astype(bool))]

def permap_pooled(run_keys):
    """Per-test-map SSE and n pooled over the given runs (seeds). Same maps for every model."""
    d = PERMAP[PERMAP["run_key"].isin(run_keys)].groupby("key")[["sse", "n", "sse_los", "n_los", "sse_nlos", "n_nlos", "sse_cov", "n_cov"]].sum()
    return d.sort_index()

def boot_rmse(pm, nb=None, seed=0, col=("sse", "n")):
    nb = nb or CFG["BOOTSTRAP"]
    s, n = pm[col[0]].to_numpy(), pm[col[1]].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(s), (nb, len(s)))
    return np.sqrt(s[idx].sum(1) / np.maximum(n[idx].sum(1), 1))

def curve_table(band):
    """One row per (model, nominal fraction): pooled RMSE, bootstrap CI, seed mean/std, LoS/NLoS."""
    H = headline(band)
    out = []
    for (m, fr), g in H.groupby(["model", "frac"]):
        pm = permap_pooled(g["run_key"])
        b = boot_rmse(pm)
        out.append(dict(model=m, family=FAMILY[m], frac=fr, frac_actual=g["frac_actual"].mean(),
                        n_maps=g["n_maps"].mean(), n_seeds=len(g),
                        rmse=math.sqrt(pm.sse.sum() / pm.n.sum()), lo=np.quantile(b, alpha_ci), hi=np.quantile(b, 1 - alpha_ci),
                        seed_mean=g["rmse"].mean(), seed_std=g["rmse"].std(ddof=1) if len(g) > 1 else np.nan,
                        rmse_los=math.sqrt(pm.sse_los.sum() / max(1, pm.n_los.sum())),
                        rmse_nlos=math.sqrt(pm.sse_nlos.sum() / max(1, pm.n_nlos.sum())),
                        rmse_cov=math.sqrt(pm.sse_cov.sum() / max(1, pm.n_cov.sum())),
                        rmse_noise_adj=math.sqrt(max(pm.sse.sum() / pm.n.sum() - NOISE_SIGMA.get(band, 0.0) ** 2, 0.0)),
                        fit_s=g["fit_s"].mean(), infer_ms=g["infer_ms"].mean()))
    return pd.DataFrame(out)

def _tex(v):
    if isinstance(v, tuple):
        v = " ".join(str(x) for x in v if str(x))
    if isinstance(v, float):
        v = "--" if not np.isfinite(v) else f"{v:.3g}"
    s = str(v)
    for a, b in [("%", "\\%"), ("_", "\\_"), ("±", "$\\pm$"), ("≤", "$\\le$"), (">=", "$\\ge$"), ("<=", "$\\le$"),
                 (">", "$>$"), ("eps1", "$\\epsilon_1$"), ("eps2", "$\\epsilon_2$"), ("eps3", "$\\epsilon_3$"), ("ν", "$\\nu$")]:
        s = s.replace(a, b)
    return s

def write_latex(df, path, caption, bold_min_cols=None):
    cols = list(df.columns)
    lines = ["\\begin{table}[t]", "\\centering", f"\\caption{{{_tex(caption)}}}", "\\begin{tabular}{" + "l" + "c" * (len(cols) - 1) + "}",
             "\\hline", " & ".join(_tex(c) for c in cols) + " \\\\", "\\hline"]
    for _, r in df.iterrows():
        lines.append(" & ".join(_tex(r[c]) for c in cols) + " \\\\")
    lines += ["\\hline", "\\end{tabular}", "\\end{table}"]
    Path(path).write_text("\n".join(lines))

def view1(band):
    ct = curve_table(band)
    ct.to_csv(PAPER_DIR / f"view1_curves_{band_tag(band)}.csv", index=False)
    fr_list = sorted(ct["frac"].unique())
    # ---- Table 1: RMSE (mean ± std over seeds) per model x fraction; best per column in bold ----
    tab = pd.DataFrame(index=[LABEL[m] for m in ORDER if m in set(ct.model)])
    for fr in fr_list:
        sub = ct[ct.frac == fr].set_index("model")
        best = sub["rmse"].idxmin()
        col = []
        for m in ORDER:
            if m not in sub.index:
                continue
            r = sub.loc[m]
            s = f"{r.rmse:.2f}" + (f" ± {r.seed_std:.2f}" if not np.isnan(r.seed_std) else "")
            col.append(f"\\textbf{{{s}}}" if m == best else s)
        tab[f"{100 * fr:g}%"] = col
    tab.to_csv(PAPER_DIR / f"table1_rmse_vs_fraction_{band_tag(band)}.csv")
    write_latex(tab.reset_index().rename(columns={"index": "Model"}), PAPER_DIR / f"table1_{band_tag(band)}.tex",
                f"Test RMSE (dB) on unseen cities vs. training fraction, {band_tag(band)} (mean $\\pm$ std over seeds).")
    # ---- LoS / NLoS split at ~10 / 50 / 100 % ----
    pick = sorted({min(fr_list, key=lambda x: abs(x - t)) for t in (0.1, 0.5, 1.0)})
    ln = ct[ct.frac.isin(pick)].pivot_table(index="model", columns="frac",
                                             values=["rmse_los", "rmse_nlos", "rmse_cov", "rmse_noise_adj"]).round(2)
    ln = ln.reindex([m for m in ORDER if m in ln.index]); ln.index = [LABEL[m] for m in ln.index]
    ln.to_csv(PAPER_DIR / f"table1b_los_nlos_{band_tag(band)}.csv")
    # ---- Figure 1 ----
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for m in ORDER:
        d = ct[ct.model == m].sort_values("frac_actual")
        if d.empty:
            continue
        x = 100 * d.frac_actual
        ax.plot(x, d.rmse, STYLE[FAMILY[m]], marker="o", ms=3.5, color=COLORS[m], label=LABEL[m])
        ax.fill_between(x, d.lo, d.hi, color=COLORS[m], alpha=0.15, lw=0)
    ax.set_xlabel("Training data (% of training maps)"); ax.set_ylabel("Test RMSE (dB), unseen cities")
    ax.set_title(f"View 1 — accuracy at fixed data, {band_tag(band)}"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); fig.savefig(PAPER_DIR / f"fig1_rmse_vs_fraction_{band_tag(band)}.pdf"); fig.savefig(PAPER_DIR / f"fig1_rmse_vs_fraction_{band_tag(band)}.png", dpi=200)
    plt.show()
    # ---- crossover f*: paired bootstrap, best deep vs best classical at each fraction ----
    H = headline(band)
    cross = []
    for fr in fr_list:
        sub = ct[ct.frac == fr]
        cl, dl = sub[sub.family == "classical"], sub[sub.family == "deep"]
        if cl.empty or dl.empty:
            continue
        mc, md = cl.loc[cl.rmse.idxmin(), "model"], dl.loc[dl.rmse.idxmin(), "model"]
        a = permap_pooled(H[(H.model == mc) & (H.frac == fr)]["run_key"])
        b = permap_pooled(H[(H.model == md) & (H.frac == fr)]["run_key"])
        a, b = a.align(b, join="inner", axis=0)
        idx = np.random.default_rng(1).integers(0, len(a), (CFG["BOOTSTRAP"], len(a)))
        diff = (np.sqrt(b.sse.to_numpy()[idx].sum(1) / b.n.to_numpy()[idx].sum(1)) -
                np.sqrt(a.sse.to_numpy()[idx].sum(1) / a.n.to_numpy()[idx].sum(1)))
        lo, hi = np.quantile(diff, [alpha_ci, 1 - alpha_ci])
        cross.append(dict(frac=fr, best_classical=LABEL[mc], best_deep=LABEL[md], diff_db=float(diff.mean()),
                          lo=float(lo), hi=float(hi),
                          verdict="deep better" if hi < 0 else ("classical better" if lo > 0 else "no significant difference")))
    cross = pd.DataFrame(cross)
    sig = cross[cross.verdict == "deep better"]
    f_star = float(sig.frac.min()) if len(sig) else None
    cross.to_csv(PAPER_DIR / f"crossover_{band_tag(band)}.csv", index=False)
    print(f"Label-noise sigma ({band_tag(band)}) = {NOISE_SIGMA[band]:.2f} dB; noise-adjusted RMSE = sqrt(RMSE^2 - sigma^2) "
          f"is reported alongside (Table 1b), so model gaps are not hidden by label noise.")
    print(f"Crossover f* ({band_tag(band)}): " + (f"{100 * f_star:g}% of training maps" if f_star else "deep never significantly better"))
    display_df(tab, "Table 1 — RMSE (dB) vs training fraction")
    display_df(ln, "Table 1b — LoS / NLoS RMSE (dB)")
    display_df(cross, "Crossover analysis (deep − classical, paired bootstrap)")
    return ct, cross, f_star

def display_df(df, title):
    print(f"\n{title}")
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())

if CFG["RUN_ANALYSIS"]:
    V1 = {f: view1(f) for f in BANDS}


### View 2: fixed target RMSE → minimum data and the data multiplier

In [ ]:
# ============================================================
# Cell 11 — Phase 5b: VIEW 2 (fixed target RMSE -> minimum data) + data multiplier + power-law fits
# ============================================================
def f_min(fr, r, eps):
    """Smallest training fraction reaching RMSE <= eps, linear interpolation in log-fraction.
    Returns (value, flag): flag '=' exact/interpolated, '<=' already met at the smallest fraction
    (left-censored), '>' never met within the data available (right-censored)."""
    fr, r = np.asarray(fr, float), np.asarray(r, float)
    if r[0] <= eps:
        return fr[0], "<="
    for i in range(1, len(fr)):
        if r[i] <= eps:
            t = (r[i - 1] - eps) / max(r[i - 1] - r[i], 1e-12)
            return float(np.exp(np.log(fr[i - 1]) + t * (np.log(fr[i]) - np.log(fr[i - 1])))), "="
    return float("inf"), ">"

def view2(band, ct):
    B = BANDS[band]
    full = ct[np.isclose(ct.frac, 1.0)].set_index("model")
    cls = [m for m in full.index if FAMILY[m] == "classical"]
    eps = {"eps1 (ridge@100%)": float(full.loc["ridge", "rmse"]) if "ridge" in full.index else None,
           "eps2 (best classical@100%)": float(full.loc[cls, "rmse"].min()),
           f"eps3 (best@100% + {CFG['EPS3_MARGIN_DB']} dB)": float(full["rmse"].min() + CFG["EPS3_MARGIN_DB"])}
    rt_h = float(B["train"]["rt_s"].median()) / 3600
    ft_h = FEAT_S_PER_MAP / 3600
    rows = []
    for en, e in eps.items():
        if e is None:
            continue
        res = {}
        for m in ORDER:
            d = ct[ct.model == m].sort_values("frac_actual")
            if d.empty:
                continue
            res[m] = dict(mean=f_min(d.frac_actual, d.rmse, e), robust=f_min(d.frac_actual, d.hi, e))
        ref_m = min([m for m in res if FAMILY[m] == "classical"], key=lambda m: res[m]["mean"][0])
        ref_v, ref_flag = res[ref_m]["mean"]
        for m, v in res.items():
            (fm, flag), (fr_rob, flag_rob) = v["mean"], v["robust"]
            mult = fm / ref_v if np.isfinite(fm) and np.isfinite(ref_v) else np.inf
            if flag == "<=" and ref_flag == "<=":
                mult_s = "both <= min"
            elif flag == ">":
                mult_s = "not reached"
            elif ref_flag == ">":                      # e.g. eps3: reached by deep models, by NO classical model
                mult_s = "no classical reaches it"
            else:
                mult_s = ("<=" if flag == "<=" else ">=" if ref_flag == "<=" else "") + f"{mult:.2f}x"
            maps = fm * B["n_train_maps_full"] if np.isfinite(fm) else np.inf
            rows.append(dict(target=en, eps_db=round(e, 3), model=m, label=LABEL[m], family=FAMILY[m],
                             f_min=fm, flag=flag, f_min_robust=fr_rob, flag_robust=flag_rob,
                             maps=maps, rt_hours=maps * rt_h, rt_feat_hours=maps * (rt_h + ft_h),
                             ref_classical=LABEL[ref_m], multiplier=mult, multiplier_str=mult_s))
    T2 = pd.DataFrame(rows)
    T2.to_csv(PAPER_DIR / f"table2_min_data_{band_tag(band)}.csv", index=False)
    show = T2.assign(f_min_pct=lambda d: [("≤" if fl == "<=" else ">" if fl == ">" else "") +
                                          (f"{100 * v:.1f}%" if np.isfinite(v) else "100%")
                                          for v, fl in zip(d.f_min, d.flag)]).pivot_table(
        index="label", columns="target", values=["f_min_pct", "multiplier_str"], aggfunc="first")
    show = show.reindex([LABEL[m] for m in ORDER if LABEL[m] in show.index])
    tex = show.copy()
    tex.columns = [("Min. data " if a == "f_min_pct" else "Multiplier ") + b.split(" ")[0] for a, b in tex.columns]
    write_latex(tex.reset_index().rename(columns={"label": "Model"}), PAPER_DIR / f"table2_{band_tag(band)}.tex",
                f"Minimum training data to reach each target RMSE, and data multiplier vs. the most data-efficient classical model ({band_tag(band)}).")
    display_df(show, "Table 2 — minimum data (% of training maps) and data multiplier")
    # ---- Figure 2: grouped bars of minimum data ----
    tg = list(eps.keys())
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    w = 0.8 / max(1, len(ORDER))
    for k, m in enumerate(ORDER):
        d = T2[T2.model == m].set_index("target").reindex(tg)
        if d.empty:
            continue
        vals = [100 * v if np.isfinite(v) else 100 * 1.05 for v in d.f_min]
        bars = ax.bar(np.arange(len(tg)) + (k - len(ORDER) / 2 + 0.5) * w, vals, w, color=COLORS[m], label=LABEL[m])
        for b_, fl in zip(bars, d.flag):
            if fl != "=":
                b_.set_hatch("//" if fl == ">" else ".."); b_.set_alpha(0.6)
    ax.set_xticks(range(len(tg))); ax.set_xticklabels([t.split(" ")[0] for t in tg])
    ax.set_ylabel("Min. training data (% of maps)"); ax.set_yscale("log")
    ax.set_title(f"View 2 — data needed for a fixed target RMSE, {band_tag(band)}\n(hatched //: not reached; ..: met at smallest fraction)", fontsize=9)
    ax.legend(fontsize=6.5, ncol=3); ax.grid(alpha=0.3, axis="y")
    fig.tight_layout(); fig.savefig(PAPER_DIR / f"fig2_min_data_{band_tag(band)}.pdf"); fig.savefig(PAPER_DIR / f"fig2_min_data_{band_tag(band)}.png", dpi=200)
    plt.show()
    # ---- optional smooth cross-check: RMSE(f) = c + a f^-b ----
    fits = []
    for m in ORDER:
        d = ct[ct.model == m].sort_values("frac_actual")
        if len(d) < 4:
            continue
        try:
            p, _ = curve_fit(lambda x, c, a, b: c + a * x ** (-b), d.frac_actual, d.rmse,
                             p0=[d.rmse.min() * 0.9, 0.5, 0.5], bounds=([0, 0, 0], [np.inf, np.inf, 3]), maxfev=20000)
            fits.append(dict(model=LABEL[m], floor_c=p[0], a=p[1], slope_b=p[2]))
        except Exception as e:
            fits.append(dict(model=LABEL[m], floor_c=np.nan, a=np.nan, slope_b=np.nan))
    fits = pd.DataFrame(fits)
    fits.to_csv(PAPER_DIR / f"powerlaw_fits_{band_tag(band)}.csv", index=False)
    display_df(fits.round(3), "Power-law fits RMSE(f) = c + a·f^(-b)")
    return T2, eps, fits

if CFG["RUN_ANALYSIS"]:
    V2 = {f: view2(f, V1[f][0]) for f in BANDS}


### Cost–benefit

In [ ]:
# ============================================================
# Cell 12 — Phase 5c: cost–benefit (complexity table + Pareto fronts)
# ============================================================
def pareto_mask(cost, err):
    o = np.argsort(cost); best, m = np.inf, np.zeros(len(cost), bool)
    for i in o:
        if err[i] < best - 1e-9:
            m[i], best = True, err[i]
    return m

def cost_table(band, ct):
    H = headline(band)
    B = BANDS[band]
    rows = []
    for m in ORDER:
        g = H[(H.model == m) & np.isclose(H.frac, 1.0)]
        if g.empty:
            continue
        c = ct[(ct.model == m) & np.isclose(ct.frac, 1.0)].iloc[0]
        data_h = g["n_maps"].mean() * (g["rt_s_per_map"].mean() + FEAT_S_PER_MAP) / 3600
        rows.append(dict(model=LABEL[m], family=FAMILY[m], rmse_100=c.rmse,
                         rmse_full_map=g["rmse_full"].mean() if "rmse_full" in g and g["rmse_full"].notna().any() else np.nan,
                         params_or_nodes=g["n_params"].mean(), size_mb=g["size_mb"].mean(),
                         gflops_per_map=(g["flops"].mean() / 1e9 if g["flops"].notna().any() else np.nan),
                         infer_ms_per_map=g["infer_ms"].mean(), fit_h=g["fit_s"].mean() / 3600,
                         tune_h=g["tune_s"].mean() / 3600, data_h=data_h,
                         total_h=data_h + g["fit_s"].mean() / 3600 + g["tune_s"].mean() / 3600,
                         cpu_peak_mb=g["cpu_peak_mb"].mean(),
                         gpu_peak_mb=(g["gpu_peak_mb"].mean() if "gpu_peak_mb" in g and g["gpu_peak_mb"].notna().any() else np.nan),
                         rows_used=(g["rows_used"].mean() if "rows_used" in g and g["rows_used"].notna().any() else np.nan),
                         cap_bound=(bool(g["cap_bound"].fillna(False).astype(bool).any()) if "cap_bound" in g else False)))
    T3 = pd.DataFrame(rows)
    T3["pareto_cost"] = pareto_mask(T3.total_h.to_numpy(), T3.rmse_100.to_numpy())
    T3["pareto_latency"] = pareto_mask(T3.infer_ms_per_map.to_numpy(), T3.rmse_100.to_numpy())
    T3.to_csv(PAPER_DIR / f"table3_cost_{band_tag(band)}.csv", index=False)
    tex = T3.assign(train_min=60 * T3.fit_h, tune_min=60 * T3.tune_h)[
        ["model", "rmse_100", "params_or_nodes", "size_mb", "gflops_per_map", "infer_ms_per_map", "train_min", "tune_min"]]
    tex.columns = ["Model", "RMSE@100% (dB)", "Params / nodes", "Size (MB)", "GFLOPs / map", "Inference (ms / map)",
                   "Train (min)", "Tune (min)"]
    write_latex(tex,
                PAPER_DIR / f"table3_{band_tag(band)}.tex", f"Accuracy and cost at 100% training data ({band_tag(band)}).")
    display_df(T3.round(3), "Table 3 — cost at 100% training data")
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, xc, xl, pm in [(axs[0], "total_h", "Total cost (h): RT+features+tuning+training", "pareto_cost"),
                           (axs[1], "infer_ms_per_map", "Inference time per map (ms)", "pareto_latency")]:
        for _, r in T3.iterrows():
            m = [k for k, v in LABEL.items() if v == r.model][0]
            ax.scatter(r[xc], r.rmse_100, color=COLORS[m], s=60 if r[pm] else 25, edgecolor="k" if r[pm] else "none", zorder=3)
            ax.annotate(r.model, (r[xc], r.rmse_100), fontsize=6.5, xytext=(3, 3), textcoords="offset points")
        p = T3[T3[pm]].sort_values(xc)
        ax.plot(p[xc], p.rmse_100, "k:", lw=1)
        ax.set_xscale("log"); ax.set_xlabel(xl); ax.set_ylabel("Test RMSE @100% (dB)"); ax.grid(alpha=0.3)
    fig.suptitle(f"Cost–benefit, {band_tag(band)} (outlined = Pareto-optimal)", fontsize=10)
    fig.tight_layout(); fig.savefig(PAPER_DIR / f"fig3_pareto_{band_tag(band)}.pdf"); fig.savefig(PAPER_DIR / f"fig3_pareto_{band_tag(band)}.png", dpi=200)
    plt.show()
    # augmentation ablation (separately labelled, never in headline)
    A = RES[(RES.band == band) & (RES["augment"].astype(bool))]
    if len(A):
        Hh = headline(band)
        ab = A.merge(Hh[["model", "frac", "seed", "rmse"]], on=["model", "frac", "seed"], suffixes=("_aug", "_noaug"))
        ab = ab[["model", "frac", "rmse_noaug", "rmse_aug"]].assign(delta_db=lambda d: d.rmse_aug - d.rmse_noaug)
        ab.to_csv(PAPER_DIR / f"ablation_d4_aug_{band_tag(band)}.csv", index=False)
        display_df(ab.round(3), "Ablation — DL with D4 augmentation (NOT part of the headline comparison)")
    return T3

if CFG["RUN_ANALYSIS"]:
    V3 = {f: cost_table(f, V1[f][0]) for f in BANDS}


### Validity checks


In [ ]:
# ============================================================
# Cell 13 — Phase 5d: label/feature validity checks, dataset table, example figure, validity checklist
# ============================================================
def los_agreement(band):
    rows = []
    for fn in sorted((ROOT / "rt_checks" / band_tag(band)).glob(f"los_*_{RT_FP}.npz")):
        sid = fn.stem[len("los_"):-len(f"_{RT_FP}")]
        m = SCENES[SCENES.scene_id == sid].iloc[0]
        bm = np.load(Path(m["dir"]) / "scene.npz")["bmask"]
        L = np.load(fn)["los"]
        for k in range(L.shape[0]):
            g = geo_los(sid, k, band)
            o = ~bm
            a, s = g[o], L[k][o]
            rows.append(dict(scene_id=sid, tx=k, agreement=float((a == s).mean()),
                             precision=float((a & s).sum() / max(1, a.sum())), recall=float((a & s).sum() / max(1, s.sum())),
                             los_frac_geo=float(a.mean()), los_frac_sionna=float(s.mean())))
    return pd.DataFrame(rows)

def k_sufficiency(band):
    fn = EXP_DIR / f"ksuff_{band_tag(band)}.json"
    if fn.exists():
        return json.load(open(fn))
    B = BANDS[band]
    hpd = json.load(open(EXP_DIR / f"hp_{band_tag(band)}_lgbm.json"))
    tr = subset_maps(B, CFG["TUNE_FRACTION"], 0)
    st = norm_stats(tr, band, 0)
    out = {}
    for k in [CFG["K_PIX"], CFG["K_PIX"] * CFG["K_CHECK_MULT"]]:
        est, info = train_once("lgbm", hpd["hp"], B, tr, st, 0, k_pix=k)
        out[str(k)] = evaluate("lgbm", est, B, st, full=False)[0]["rmse"]
    out["delta_db"] = out[str(CFG["K_PIX"])] - out[str(CFG["K_PIX"] * CFG["K_CHECK_MULT"])]
    json.dump(out, open(fn, "w"), indent=1)
    return out

def dataset_table(band):
    B = BANDS[band]
    rows = []
    for s in ["train", "val", "test"]:
        mm = RT_INDEX[(RT_INDEX.band == band) & (RT_INDEX.split == s)]
        sc = SCENES[SCENES.split == s]
        rows.append(dict(split=s, cities=", ".join(sorted(sc.city.unique())), scenes=len(sc), maps_total=len(mm),
                         maps_included=int(mm.include.sum()), bldg_cover=sc.cover.mean(),
                         height_median_m=sc.height_median.mean(), height_imputed_frac=sc.imputed_frac.mean(),
                         unhit_px_frac=mm.invalid_frac.mean(), at_floor_px_frac=mm.floor_frac.mean(),
                         pl_median_db=mm.pl_median.median(),
                         rt_s_per_map=mm.rt_s.median(), feat_s_per_map=FEAT_S_PER_MAP))
    T0 = pd.DataFrame(rows)
    T0.to_csv(PAPER_DIR / f"table0_dataset_{band_tag(band)}.csv", index=False)
    return T0

def example_figure(band):
    B = BANDS[band]
    r = B["test"].iloc[0]
    x = load_x(r["scene_id"], r["tx_idx"], band)
    y, v = load_y(r)
    tx = SCENES[SCENES.scene_id == r["scene_id"]].iloc[0]["tx"][r["tx_idx"]]
    ext = [-T / 2, T / 2, -T / 2, T / 2]
    show = [("Building height (m)", x[FEATS.index("height")] if "height" in FEATS else None),
            ("Path loss (dB), Sionna RT", np.where(v, y, np.nan)),
            ("Geometric LoS", x[FEATS.index("los")] if "los" in FEATS else None),
            ("Obstruction depth (m)", x[FEATS.index("depth")] if "depth" in FEATS else None),
            ("Knife-edge ν", x[FEATS.index("nu")] if "nu" in FEATS else None)]
    show = [s for s in show if s[1] is not None]
    fig, axs = plt.subplots(1, len(show), figsize=(3.1 * len(show), 3.0))
    for ax, (t, im) in zip(np.atleast_1d(axs), show):
        h = ax.imshow(im, origin="lower", extent=ext, cmap="viridis")
        ax.plot(tx[0], tx[1], "r^", ms=6); ax.set_title(t, fontsize=8); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(h, ax=ax, fraction=0.046)
    fig.suptitle(f"Example test tile {r['scene_id']} (unseen city), Tx {r['tx_idx']}, {band_tag(band)}", fontsize=9)
    fig.tight_layout(); fig.savefig(PAPER_DIR / f"fig0_example_{band_tag(band)}.png", dpi=200); fig.savefig(PAPER_DIR / f"fig0_example_{band_tag(band)}.pdf")
    plt.show()

def validity_checklist(band):
    B = BANDS[band]
    H, A = headline(band), RES[(RES.band == band) & (RES["augment"].astype(bool))]
    chk = []
    def add(name, ok, detail, level="FAIL"):
        chk.append(dict(check=name, status="PASS" if ok else level, detail=detail))
    sc = {s: set(SCENES[SCENES.split == s].city) for s in ["train", "val", "test"]}
    add("P4 city-level split disjoint", not (sc["train"] & sc["test"] or sc["train"] & sc["val"] or sc["val"] & sc["test"]),
        f"train={sorted(sc['train'])} val={sorted(sc['val'])} test={sorted(sc['test'])}")
    nested = all(set(train_scene_subset(B["train_scenes"], a, s)) <= set(train_scene_subset(B["train_scenes"], b, s))
                 for s in set(CFG["SEEDS_SMALL"]) | set(CFG["SEEDS_LARGE"])
                 for a, b in zip(sorted(CFG["FRACTIONS"])[:-1], sorted(CFG["FRACTIONS"])[1:]))
    add("P6 nested training subsets", nested, "subset(f_i) ⊂ subset(f_j) for every seed and f_i < f_j")
    add("P1 identical feature stack for all models", H.feats_hash.nunique() == 1,
        f"feature hash(es): {H.feats_hash.unique().tolist()}, channels={FEATS}")
    add("P8 identical test pixels for all models", H.eval_hash.nunique() == 1 and H.eval_hash.iloc[0] == B["eval_hash"],
        f"eval-pixel hash {B['eval_hash']}, {CFG['EVAL_PIX_PER_MAP']} px/map x {len(B['test'])} test maps")
    r0 = H.iloc[len(H) // 2]
    st0 = norm_stats(subset_maps(B, r0.frac, int(r0.seed)), band, int(r0.seed))
    add("P1 normalisation from training subset only", stats_hash(st0) == r0.stats_hash,
        f"recomputed stats hash for {r0.run_key} matches the logged one")
    add("P9 headline runs without augmentation", (~H["augment"].astype(bool)).all() and (A.empty or A["augment"].astype(bool).all()),
        f"{len(H)} headline runs, {len(A)} separately-labelled augmentation-ablation runs")
    add("P7 equal tuning budget", H.tune_trials.nunique() == 1, f"trials per model = {H.tune_trials.unique().tolist()}")
    exp = {(m, fr, s) for m in ALL_MODELS for fr in CFG["FRACTIONS"] for s in seeds_for(fr)}
    got = set(zip(H.model, H.frac, H.seed))
    add("Grid complete (model x fraction x seed)", exp <= got, f"missing: {sorted(exp - got)[:6]}")
    ncl = len(set(H.model) & set(CLASSICAL))
    add("P3 >= 5 classical learners", ncl >= 5, f"{ncl} classical: {sorted(set(H.model) & set(CLASSICAL))}")
    add("P2 no untrained baselines", all(m in FAMILY for m in ALL_MODELS), f"models: {ALL_MODELS}")
    add("P5 data counted in maps", (H.n_maps <= H.n_scenes * CFG["N_TX_PER_SCENE"]).all() and "frac_actual" in H,
        "n_maps and frac_actual logged per run; classical pixels drawn from the same maps")
    nf = NOISE[band]
    add("P13 label-noise floor measured", len(nf) > 0,
        (f"label sigma = {NOISE_SIGMA[band]:.2f} dB (median |diff| {nf.median_abs_diff.median():.2f} dB, "
         f"p95 {nf.p95_abs_diff.median():.1f} dB) over {len(nf)} maps") if len(nf) else "no noise-check maps")
    la = los_agreement(band)
    add("P13 geometric LoS vs Sionna LoS", len(la) > 0 and la.agreement.mean() >= CFG["LOS_AGREEMENT_MIN"],
        f"agreement = {la.agreement.mean():.3f} (threshold {CFG['LOS_AGREEMENT_MIN']})" if len(la) else "no LoS-check maps", level="WARN")
    ks = k_sufficiency(band)
    add("P14 K pixels per map sufficient", abs(ks["delta_db"]) < CFG["K_SUFFICIENCY_DB"],
        f"LightGBM RMSE K={CFG['K_PIX']}: {ks[str(CFG['K_PIX'])]:.3f} dB vs {CFG['K_CHECK_MULT']}K: "
        f"{ks[str(CFG['K_PIX'] * CFG['K_CHECK_MULT'])]:.3f} dB (Δ={ks['delta_db']:+.3f})", level="WARN")
    mm = RT_INDEX[RT_INDEX.band == band]
    exc = mm[~mm.include] if CFG["LABEL_MODE"] == "mask" else mm.iloc[:0]
    add("Label policy for unhit/weak cells logged", True,
        f"{LABEL_TAG}: mean unhit outdoor cells {mm.invalid_frac.mean():.1%}, at floor {mm.floor_frac.mean():.1%}; "
        f"{len(exc)} maps excluded")
    add("P15 fingerprints + versions saved", (EXP_DIR / "config_snapshot.json").exists(),
        f"SCENE={SCENE_FP} RT={RT_FP} FEAT={FEAT_FP} EXP={EXP_FP}")
    cl = pd.DataFrame(chk)
    cl.to_csv(PAPER_DIR / f"validity_checklist_{band_tag(band)}.csv", index=False)
    nf.to_csv(PAPER_DIR / f"label_noise_{band_tag(band)}.csv", index=False)
    la.to_csv(PAPER_DIR / f"los_agreement_{band_tag(band)}.csv", index=False)
    return cl, nf, la, ks

if CFG["RUN_ANALYSIS"]:
    for f in BANDS:
        display_df(dataset_table(f), f"Table 0 — dataset summary, {band_tag(f)}")
        example_figure(f)
        CL, NF, LA, KS = validity_checklist(f)
        best = V1[f][0][np.isclose(V1[f][0].frac, 1.0)].rmse.min()
        sg = NOISE_SIGMA[f]
        print(f"Label-noise floor: sigma = {sg:.2f} dB vs best test RMSE {best:.2f} dB -> label noise explains "
              f"{100 * sg ** 2 / best ** 2:.0f}% of the best model's MSE"
              + ("" if sg < 0.5 * best else "  (CHECK: consider raising RT samples_per_tx)"))
        display_df(CL, f"Validity checklist — {band_tag(f)}")
        n_fail = (CL.status == "FAIL").sum()
        print(f"\n{'ALL CHECKS PASSED' if n_fail == 0 else f'{n_fail} CHECK(S) FAILED'} "
              f"({(CL.status == 'WARN').sum()} warning(s)). Outputs in {PAPER_DIR}")


In [ ]:
# ============================================================
# Cell 14 — Phase 5e: covered-pixel analysis + coverage-hole detection   (run AFTER cells 10–13)
#   (a) error split: all / covered (label < floor) / floor pixels, per model x fraction
#   (b) View 1 + View 2 + cost table re-run on COVERED pixels  -> PAPER_DIR/covered_pixels/
#   (c) coverage-hole detection (F1 at a max-allowable path loss, AUROC for "no RT path") from the SAVED models
# ============================================================
from sklearn.metrics import roc_auc_score

COV_MAPL_DB = [140.0, 146.0, 150.0]   # max allowable path loss; 146 dB ~ 46 dBm EIRP with a -100 dBm receiver
COV_MAPL_MAIN = 146.0
COV_FRACS = [f for f in [0.05, 0.1, 0.3, 1.0] if any(np.isclose(f, x) for x in CFG["FRACTIONS"])] or CFG["FRACTIONS"]
COV_SEEDS = [0]                       # add 1, 2 for seed spread at <= 30 % (kNN / SVR re-prediction is slow)
COV_LOG = EXP_DIR / "coverage_eval.jsonl"

_PERMAP_ALL = PERMAP.copy()
_PERMAP_ALL["sse_flr"] = _PERMAP_ALL["sse"] - _PERMAP_ALL["sse_cov"]
_PERMAP_ALL["n_flr"] = _PERMAP_ALL["n"] - _PERMAP_ALL["n_cov"]
METRIC_COLS = {"all": ("sse", "n"), "cov": ("sse_cov", "n_cov"), "floor": ("sse_flr", "n_flr")}
METRIC = "all"

def permap_pooled(run_keys):
    """Per-test-map SSE and n pooled over runs; 'sse'/'n' hold the ACTIVE metric (all / cov / floor)."""
    d = _PERMAP_ALL[_PERMAP_ALL["run_key"].isin(run_keys)].groupby("key")[
        ["sse", "n", "sse_los", "n_los", "sse_nlos", "n_nlos", "sse_cov", "n_cov", "sse_flr", "n_flr"]].sum()
    s, n = METRIC_COLS[METRIC]
    d["sse"], d["n"] = d[s].copy(), d[n].copy()
    return d.sort_index()

def curve_table(band):
    """As in cell 10, but the seed mean/std are recomputed from per-map stats so they follow the active metric."""
    H = headline(band)
    out = []
    for (m, fr), g in H.groupby(["model", "frac"]):
        pm = permap_pooled(g["run_key"])
        b = boot_rmse(pm)
        per_seed = [math.sqrt(p.sse.sum() / max(1, p.n.sum())) for p in (permap_pooled([k]) for k in g["run_key"])]
        out.append(dict(model=m, family=FAMILY[m], frac=fr, frac_actual=g["frac_actual"].mean(),
                        n_maps=g["n_maps"].mean(), n_seeds=len(g),
                        rmse=math.sqrt(pm.sse.sum() / max(1, pm.n.sum())), lo=np.quantile(b, alpha_ci), hi=np.quantile(b, 1 - alpha_ci),
                        seed_mean=float(np.mean(per_seed)), seed_std=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else np.nan,
                        rmse_los=math.sqrt(pm.sse_los.sum() / max(1, pm.n_los.sum())),
                        rmse_nlos=math.sqrt(pm.sse_nlos.sum() / max(1, pm.n_nlos.sum())),
                        rmse_cov=math.sqrt(pm.sse_cov.sum() / max(1, pm.n_cov.sum())),
                        rmse_noise_adj=math.sqrt(max(pm.sse.sum() / max(1, pm.n.sum()) - NOISE_SIGMA.get(band, 0.0) ** 2, 0.0)),
                        fit_s=g["fit_s"].mean(), infer_ms=g["infer_ms"].mean()))
    return pd.DataFrame(out)

# ---------------- (a) error split table ----------------
def error_split(band):
    H = headline(band)
    rows = []
    for (m, fr), g in H.groupby(["model", "frac"]):
        d = _PERMAP_ALL[_PERMAP_ALL["run_key"].isin(g["run_key"])][["sse", "n", "sse_cov", "n_cov", "sse_flr", "n_flr"]].sum()
        rows.append(dict(model=m, frac=fr, rmse_all=math.sqrt(d.sse / d.n), rmse_cov=math.sqrt(d.sse_cov / max(1, d.n_cov)),
                         rmse_floor=math.sqrt(d.sse_flr / max(1, d.n_flr)), floor_pct=100 * d.n_flr / d.n))
    E = pd.DataFrame(rows)
    E.to_csv(PAPER_DIR / f"table1c_error_split_{band_tag(band)}.csv", index=False)
    print(f"Floor pixels (label = {CFG['PL_FLOOR_DB']:g} dB, no RT path): {E.floor_pct.median():.1f}% of evaluation pixels")
    for col, name in [("rmse_all", "ALL"), ("rmse_cov", "COVERED"), ("rmse_floor", "FLOOR")]:
        p = E.pivot_table(index="model", columns="frac", values=col).reindex([m for m in ORDER if m in set(E.model)])
        p.index = [LABEL[m] for m in p.index]
        p.columns = [f"{100 * c:g}%" for c in p.columns]
        display_df(p.round(2), f"Table 1c — RMSE on {name} pixels (dB, pooled over seeds), {band_tag(band)}")
    return E

# ---------------- (c) coverage-hole detection from saved models ----------------
def eval_pixel_preds(row, B):
    """Re-predict the fixed test pixels with a saved model. Returns (pred_dB, label_dB, test-map index)."""
    est, meta = load_saved_model(row["model_path"])
    seed = int(row["seed"])
    st = norm_stats(subset_maps(B, row["frac"], seed), B["f"], seed)
    if stats_hash(st) != row["stats_hash"]:
        print(f"   WARNING {row['run_key']}: normalisation hash differs from the logged run")
    test, P, Y, K = B["test"], [], [], []
    if FAMILY[row["model"]] == "classical":
        X, _, _ = rows_at_pixels(test, B["eval_pix"], st, B["f"])
        pred, off = predict_classical(est, X) * st["ysd"] + st["ymu"], 0
        for i, (_, r) in enumerate(test.iterrows()):
            idx = B["eval_pix"][r["key"]]; n = len(idx)
            y, _ = load_y(r)
            P.append(pred[off:off + n]); Y.append(y.ravel()[idx]); K.append(np.full(n, i)); off += n
    else:
        with torch.no_grad():
            for i, (r, pdb) in enumerate(deep_predict_maps(est, test, st, B["f"])):
                idx = B["eval_pix"][r["key"]]
                y, _ = load_y(r)
                P.append(pdb.ravel()[idx]); Y.append(y.ravel()[idx]); K.append(np.full(len(idx), i))
    del est
    return np.concatenate(P), np.concatenate(Y), np.concatenate(K)

def coverage_metrics(p, y, k):
    out, nm = {}, int(k.max()) + 1
    nopath = y >= CFG["PL_FLOOR_DB"] - 1e-3
    out["nopath_pct"] = 100 * float(nopath.mean())
    out["auc_nopath"] = float(roc_auc_score(nopath, p)) if 0 < nopath.mean() < 1 else np.nan
    idx = np.random.default_rng(2).integers(0, nm, (CFG["BOOTSTRAP"], nm))
    for T in COV_MAPL_DB:
        t, h = y >= T, p >= T
        TP = np.bincount(k, weights=(t & h).astype(float), minlength=nm)
        FP = np.bincount(k, weights=(~t & h).astype(float), minlength=nm)
        FN = np.bincount(k, weights=(t & ~h).astype(float), minlength=nm)
        tp, fp, fn = TP.sum(), FP.sum(), FN.sum()
        f1b = 2 * TP[idx].sum(1) / np.maximum(1, 2 * TP[idx].sum(1) + FP[idx].sum(1) + FN[idx].sum(1))
        out.update({f"hole_pct_{T:g}": 100 * float(t.mean()), f"f1_{T:g}": float(2 * tp / max(1, 2 * tp + fp + fn)),
                    f"prec_{T:g}": float(tp / max(1, tp + fp)), f"rec_{T:g}": float(tp / max(1, tp + fn)),
                    f"f1_{T:g}_lo": float(np.quantile(f1b, alpha_ci)), f"f1_{T:g}_hi": float(np.quantile(f1b, 1 - alpha_ci))})
    return out

def coverage_phase(band):
    B, H = BANDS[band], headline(band)
    done = set(pd.read_json(COV_LOG, lines=True)["run_key"]) if COV_LOG.exists() else set()
    sel = H[H["frac"].apply(lambda v: any(np.isclose(v, f) for f in COV_FRACS)) & H["seed"].isin(COV_SEEDS)]
    sel = sel.assign(_o=sel["model"].map({m: i for i, m in enumerate(ORDER)})).sort_values(["_o", "frac"])
    todo = [r for _, r in sel.iterrows() if r["run_key"] not in done]
    print(f"Coverage-hole evaluation: {len(sel)} runs selected, {len(todo)} to do (resumable, log: {COV_LOG.name})")
    for n, row in enumerate(todo):
        if not isinstance(row.get("model_path"), str) or not Path(row["model_path"]).exists():
            print(f"   skip {row['run_key']}: no saved model"); continue
        t0 = time.time()
        p, y, k = eval_pixel_preds(row, B)
        rec = dict(run_key=row["run_key"], band=band, model=row["model"], frac=float(row["frac"]), seed=int(row["seed"]),
                   **coverage_metrics(p, y, k))
        _append(COV_LOG, rec)
        print(f"   [{n + 1}/{len(todo)}] {LABEL[row['model']]:>15s} f={row['frac']:<5g}: F1@{COV_MAPL_MAIN:g} "
              f"{rec[f'f1_{COV_MAPL_MAIN:g}']:.3f}, AUROC(no path) {rec['auc_nopath']:.3f} ({time.time() - t0:.0f}s)")
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    C = pd.read_json(COV_LOG, lines=True, precise_float=True)
    C["frac"] = C["frac"].round(6)
    C = C[C["run_key"].isin(H["run_key"])]
    C.to_csv(PAPER_DIR / f"coverage_holes_{band_tag(band)}.csv", index=False)
    order = [m for m in ORDER if m in set(C.model)]
    tabs = {}
    for col, name in [(f"f1_{COV_MAPL_MAIN:g}", f"F1 for coverage holes (PL >= {COV_MAPL_MAIN:g} dB)"),
                      ("auc_nopath", "AUROC for 'no RT path' pixels")]:
        t = C.pivot_table(index="model", columns="frac", values=col, aggfunc="mean").reindex(order)
        t.index = [LABEL[m] for m in t.index]; t.columns = [f"{100 * c:g}%" for c in t.columns]
        tabs[col] = t
        display_df(t.round(3), f"Table 4 — {name}, {band_tag(band)} (seed(s) {COV_SEEDS})")
    s100 = C[np.isclose(C.frac, C.frac.max())]
    sens = s100.set_index("model")[[f"f1_{T:g}" for T in COV_MAPL_DB] + [f"hole_pct_{T:g}" for T in COV_MAPL_DB]]
    sens = sens.groupby(level=0).mean().reindex(order); sens.index = [LABEL[m] for m in sens.index]
    display_df(sens.round(3), f"Coverage-hole F1 vs threshold at {100 * C.frac.max():g}% data (hole_pct = share of pixels that are holes)")
    tex = tabs[f"f1_{COV_MAPL_MAIN:g}"].round(3).reset_index().rename(columns={"index": "Model"})
    write_latex(tex, PAPER_DIR / f"table4_coverage_f1_{band_tag(band)}.tex",
                f"Coverage-hole detection F1 (PL >= {COV_MAPL_MAIN:g} dB) on unseen cities vs. training fraction ({band_tag(band)}).")
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for m in order:
        d = C[C.model == m].groupby("frac")[[f"f1_{COV_MAPL_MAIN:g}", f"f1_{COV_MAPL_MAIN:g}_lo", f"f1_{COV_MAPL_MAIN:g}_hi"]].mean()
        ax.errorbar(100 * d.index, d.iloc[:, 0], yerr=[d.iloc[:, 0] - d.iloc[:, 1], d.iloc[:, 2] - d.iloc[:, 0]],
                    fmt=STYLE[FAMILY[m]] + "o", ms=3.5, capsize=2, color=COLORS[m], label=LABEL[m])
    ax.set_xscale("log"); ax.set_xlabel("Training data (% of training maps)"); ax.set_ylabel(f"F1, coverage holes (PL ≥ {COV_MAPL_MAIN:g} dB)")
    ax.set_title(f"Coverage-hole detection on unseen cities, {band_tag(band)}"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); fig.savefig(PAPER_DIR / f"fig4_coverage_f1_{band_tag(band)}.pdf"); fig.savefig(PAPER_DIR / f"fig4_coverage_f1_{band_tag(band)}.png", dpi=200)
    plt.show()
    return C

if CFG["RUN_ANALYSIS"]:
    _PAPER_BASE = PAPER_DIR
    E_SPLIT, V1C, V2C, V3C, COV = {}, {}, {}, {}, {}
    for f in BANDS:
        print(f"\n################ (a) error split, {band_tag(f)} ################")
        E_SPLIT[f] = error_split(f)
        print(f"\n################ (b) COVERED pixels only (label < {CFG['PL_FLOOR_DB']:g} dB), {band_tag(f)} ################")
        METRIC, PAPER_DIR = "cov", _PAPER_BASE / "covered_pixels"
        PAPER_DIR.mkdir(exist_ok=True)
        try:
            V1C[f] = view1(f)                 # figure axes say "Test RMSE": here it is covered-pixel RMSE
            V2C[f] = view2(f, V1C[f][0])
            V3C[f] = cost_table(f, V1C[f][0])
        finally:
            METRIC, PAPER_DIR = "all", _PAPER_BASE
        print(f"\n################ (c) coverage-hole detection, {band_tag(f)} ################")
        COV[f] = coverage_phase(f)
    print(f"\nOutputs: {PAPER_DIR} (all pixels, coverage) and {PAPER_DIR / 'covered_pixels'} (covered-pixel views)")

### Summary tables used in the paper


In [ ]:
# ============================================================
# Cell 15 — Table I values (RMSE over all and covered pixels for every training size)
# ============================================================
for f in BANDS:
    for name, sub in [("all pixels", ""), ("covered pixels", "covered_pixels")]:
        c = pd.read_csv(PAPER_DIR / sub / f"view1_curves_{band_tag(f)}.csv")
        c = c.assign(maps=c["n_maps"].round().astype(int))            # actual number of training maps
        rows = [m for m in ORDER if m in set(c["model"])]
        t = c.pivot_table(index="model", columns="maps", values="rmse").reindex(rows)
        t.index = [LABEL[m] for m in t.index]
        print(f"\nTest RMSE (dB), {name}, {band_tag(f)}\n{t.round(2).to_string()}")
        s = c.pivot_table(index="model", columns="maps", values="seed_std").reindex(rows).dropna(axis=1, how="all")
        s.index = [LABEL[m] for m in s.index]
        print(f"\nSeed-to-seed standard deviation (dB), {name}\n{s.round(2).to_string()}")


### Tuned hyperparameters


In [ ]:
# ============================================================
# Cell 16 — Settings selected by the random search (frozen for all training sizes and seeds)
# ============================================================
for fn in sorted(EXP_DIR.glob("hp_*.json")):
    d = json.load(open(fn))
    print(f"{LABEL[d['model']]:>12s}  {d['hp']}  (validation RMSE {d['val_rmse']:.2f} dB)")


## Reproducing the results

1. **Test run.** Set `DRY_RUN = True` and run all cells. The pipeline finishes on a CPU in a few minutes and writes to `rm_ml_vs_dl_dryrun/`. All checklist items should print PASS.
2. **Full run.**
   - Set `DRY_RUN = False` and select a GPU runtime. The paper used a High-RAM runtime (polynomial ridge regression needs about 16 GB of RAM at 1600 maps).
   - Results are written to `RM_DRIVE_ROOT` (default `/content/drive/MyDrive/rm_ml_vs_dl`).
   - Phase 1 downloads building footprints from the Overpass API and caches them in `osm_cache/`.
   - Phases 1–3 are resumable and skip finished scenes, maps and feature files. Ray tracing takes about 0.3 s per map on a T4.
3. **GPU memory.** Sionna (Dr.Jit) and PyTorch share the GPU. If the experiment phase runs out of memory directly after ray tracing, restart the runtime, set `RUN_SCENES = RUN_RT = RUN_FEATURES = False` and run again. Everything reloads from Drive.
4. **Resuming.** Every finished run is appended to `results.jsonl`, and every tuning trial to `tuning_trials/`, so the experiment phase can be interrupted and resumed.
5. **Optional extensions (not used in the paper).** `FEATURE_SET = "A"` (reduced feature set), `OPTIONAL_MODELS = ["unet_shallow"]`, `RUN_AUG_ABLATION = True` (D4 augmentation) and additional bands in `BANDS_GHZ`. Each creates its own experiment fingerprint and reuses the existing scenes.

## Output files (`exp/<fingerprint>/paper/`)

| File | Content |
|---|---|
| `table0_dataset_*.csv` | Dataset summary per split: cities, tiles, maps, building cover, heights, imputation rate, ray-tracing and feature time per map |
| `fig0_example_*.pdf` | Example test tile: building heights, ray-traced path loss and three feature channels |
| `view1_curves_*.csv`, `table1_*.tex`, `fig1_*.pdf` | RMSE versus training data for every model (bootstrap confidence intervals, seed spread) |
| `table1b_los_nlos_*.csv`, `table1c_error_split_*.csv` | LoS/NLoS and covered/floor error breakdowns |
| `crossover_*.csv` | Paired-bootstrap difference between the best spatial and the best classical model at each size |
| `table2_*.tex`, `table2_min_data_*.csv` | Minimum number of maps per target RMSE and data multiplier |
| `table3_*.tex`, `table3_cost_*.csv`, `fig3_*.pdf` | Complexity and cost, Pareto fronts |
| `coverage_holes_*.csv`, `table4_*.tex`, `fig4_*.pdf` | Coverage-hole F1 (140, 146 and 150 dB) and AUROC for unreached pixels |
| `covered_pixels/` | The same accuracy, data and cost analyses on covered pixels only |
| `validity_checklist_*.csv`, `label_noise_*.csv`, `los_agreement_*.csv` | Automated PASS/FAIL checks, label noise and LoS-feature agreement |
